# Banking77 — Intent Classification

We take one dataset — `PolyAI/banking77`, 77 fine-grained customer-service intents — and solve it four
times, each time with the dominant technique of a different era of NLP:

| Generation | Representation of text | What the model can learn |
|---|---|---|
| 1. TF-IDF + Logistic Regression | sparse bag-of-n-grams | which *words* signal which intent |
| 2. SimpleRNN | learned dense embeddings, read left→right | word order, but only over a short memory |
| 3. BiLSTM | learned dense embeddings + gated memory, both directions | longer dependencies, context from both sides |
| 4. DistilBERT (fine-tuned) | pretrained subword embeddings + self-attention | language knowledge transferred from a huge corpus |

The point of the notebook is **not** that we train four models. It is that at each step you can see
*what the previous generation could not represent*, and *what the new one bought us — and at what cost*.

---

### Execution record

Sections 1–6 and 8–13 contain their original executed outputs. The DistilBERT experiment was then
completed with the reproducible companion script `run_distilbert_cpu.py`, using the same official
Banking77 train/test data and random seed. That verified run achieved **0.9039 test accuracy** and
**0.9040 test macro F1**. Its full configuration and metrics are stored in
`artifacts/distilbert/metrics.json`.

The Section 7 cells remain GPU-ready and automatically select CUDA on Google Colab. Their inline
outputs are intentionally left empty because the recorded companion run used a separate stratified
80/20 train/validation split; results from two different validation splits should not be presented
as if they came from the same notebook execution.

### How to read each section

Every major section opens with the same four questions:

> **What are we doing? · Why are we doing it? · What is the input? · What is the output?**

---

### Environment

Executed with: Python 3.11, `scikit-learn` 1.8, `pandas` 3.0, `numpy` 2.4, `tensorflow-cpu` 2.21.
Section 7 additionally needs `transformers>=4.40`, `torch>=2.0`, `datasets>=2.18`, `accelerate>=0.30`.
Version-sensitive spots are called out in comments where the API has changed across releases.


## Section 0 — Setup and reproducibility

**What:** imports, global seeds, and the two constants that govern the whole experiment.
**Why:** an ML notebook that gives a different answer on every run cannot be studied or debugged.
Reproducibility is not a nicety here — the whole notebook is an *argument* that model B beats model A,
and that argument is worthless if the numbers move by 3 points when you re-run it.
**Input:** nothing. **Output:** a deterministic environment.


In [ ]:
import os
                                                                                               
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json
import random
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

                                                                             
                  
                                                                            
                                                                                
                                                                             
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

                                                                        
                                                                               
VALIDATION_FRACTION = 0.15

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Seed:", SEED)
print("Validation fraction (of training set):", VALIDATION_FRACTION)

### Imports and Configuration

In [ ]:
                                                                                   
                                                                                              
import sklearn
import matplotlib

print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"matplotlib   {matplotlib.__version__}")

                                                                          
                                                                                       
try:
    import tensorflow as tf
    TF_AVAILABLE = True
    print(f"tensorflow   {tf.__version__}")
except ImportError:
    TF_AVAILABLE = False
    print("tensorflow   NOT INSTALLED  ->  Sections 5-6 will be skipped")

try:
    import torch
    import transformers
    TRANSFORMERS_AVAILABLE = True
    print(f"torch        {torch.__version__}")
    print(f"transformers {transformers.__version__}")
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("transformers NOT INSTALLED  ->  Section 7 will be skipped")
    print("             install with: pip install 'transformers>=4.40' 'torch>=2.0' 'datasets>=2.18' 'accelerate>=0.30'")

---

## Section 1 — Problem definition and dataset understanding

**What:** define the task, load Banking77, and look at what one row actually is.
**Why:** every later decision — sequence length, vectoriser, metric — is downstream of facts about
the data. Choosing a model before reading the data is how people end up optimising the wrong thing.
**Input:** the raw dataset. **Output:** `train_df`, `test_df`, and a fixed label ↔ id mapping.

### The business problem

A retail bank's support channel receives free-text customer messages:

> *"I think my card is still on its way, it's been two weeks"*
> *"Why was I charged extra on my statement?"*
> *"The exchange rate you used looks wrong"*

Before anything useful can happen — routing to the right queue, firing the right self-service flow,
answering with the right article — the system has to decide **what the customer is asking about**.
That decision is the *intent*.

This is **single-label multiclass classification with 77 classes**: exactly one intent per message.

### Why 77 classes makes this hard, and interesting

Most tutorial text classification is 2–5 broad, well-separated classes. Banking77 is the opposite,
and it is deliberately the *realistic* case:

1. **Fine granularity.** Not "card problem" but `card_arrival`, `card_not_working`,
   `card_delivery_estimate`, `card_about_to_expire`, `lost_or_stolen_card` — five different
   operational outcomes hiding behind one topic.
2. **Semantic near-neighbours.** `card_payment_fee_charged` vs `transaction_charged_twice` vs
   `extra_charge_on_statement` all mean "money left my account and I don't like it". The *words*
   overlap heavily; only the precise complaint differs.
3. **Short inputs.** ~12 words on average. There is very little context to disambiguate with.
4. **Modest training data.** ~10k examples spread over 77 classes ≈ 130 examples per class. This is
   the regime where transfer learning from a pretrained model should pay off most.

### What the target represents

The target is **not** a sentiment, a priority, or a department. It is the *specific customer goal*,
drawn from a closed, hand-curated taxonomy of 77 goals that this bank's operations team can act on.
The label set is **flat** (no hierarchy) and **mutually exclusive** — which is itself a modelling
assumption, and one we will see the model quietly punished for in the error analysis.


In [ ]:
                                                                             
                    
 
                                                    
                                                                          
                                                                             
                                           
 
                                                                                  
                                                                                  
                                                                             
                                               
                                                                             
GITHUB_BASE = ("https://raw.githubusercontent.com/PolyAI-LDN/"
               "task-specific-datasets/master/banking_data")

def load_banking77():
    """Return (train_df, test_df) with columns ['text', 'category'] and the source used."""
    try:
        from datasets import load_dataset
        ds = load_dataset("PolyAI/banking77")
                                                                                        
        label_names = ds["train"].features["label"].names
        to_df = lambda split: pd.DataFrame({
            "text": ds[split]["text"],
            "category": [label_names[i] for i in ds[split]["label"]],
        })
        return to_df("train"), to_df("test"), "huggingface:PolyAI/banking77"
    except Exception as exc:                                                              
        print(f"[info] Hugging Face load failed ({type(exc).__name__}); using the authors' CSVs.")
        train = pd.read_csv(f"{GITHUB_BASE}/train.csv")
        test = pd.read_csv(f"{GITHUB_BASE}/test.csv")
        return train, test, "github:PolyAI-LDN/task-specific-datasets"

train_df, test_df, DATA_SOURCE = load_banking77()

print("source          :", DATA_SOURCE)
print("train_df.shape  :", train_df.shape)
print("test_df.shape   :", test_df.shape)
print("columns         :", list(train_df.columns))
print("n intents (train):", train_df["category"].nunique())
print("n intents (test) :", test_df["category"].nunique())

### Results

In [ ]:
                                                                             
                           
 
                                                                                   
                                                                              
                                                                                    
                               
 
                                                                              
                                         
                                                                             
INTENT_NAMES = sorted(train_df["category"].unique())
NUM_CLASSES = len(INTENT_NAMES)

name_to_id = {name: i for i, name in enumerate(INTENT_NAMES)}
id_to_name = {i: name for name, i in name_to_id.items()}

                                                                                 
unseen = set(test_df["category"]) - set(INTENT_NAMES)
assert not unseen, f"test set contains intents absent from train: {unseen}"

print(f"NUM_CLASSES = {NUM_CLASSES}\n")
print("First 15 intents:")
for i in range(15):
    print(f"  {i:2d}  {INTENT_NAMES[i]}")
print("  ...")
print(f"  {NUM_CLASSES-1:2d}  {INTENT_NAMES[-1]}")

### Results

In [ ]:
                                                                                      
                                                                              
pd.set_option("display.max_colwidth", 90)

sample_intents = ["card_arrival", "exchange_rate", "lost_or_stolen_card",
                  "topping_up_by_card", "why_verify_identity"]

for intent in sample_intents:
    subset = train_df.loc[train_df["category"] == intent, "text"]
    print(f"--- {intent}  ({len(subset)} training examples) ---")
    for text in subset.head(3):
        print("   ", text)
    print()

---

## Section 2 — NLP-specific exploratory data analysis

**What:** measure the properties of the text that will actually change our modelling choices.
**Why:** generic EDA (`df.describe()`) tells you nothing about text. The questions that matter here
are: *how long are the inputs* (→ sequence length), *how imbalanced are the classes* (→ which metric),
*how much do classes overlap in vocabulary* (→ how hard is the ceiling), *is there leakage or
duplication* (→ are my scores real).
**Input:** `train_df`. **Output:** numbers and plots that justify Sections 3–7.

> ### The rule we follow for the whole of Section 2
> **Every statistic here is computed on the training data only.**
> Looking at the test set — even just to plot its length distribution — leaks information from it
> into your decisions. You would be choosing a sequence length, a vocabulary size, or a
> preprocessing step *informed by the test set*, and your final number stops being an honest
> estimate of performance on unseen data. We compute the test statistics exactly once, in Section 9,
> after everything is frozen.


In [ ]:
                                                                                

print("Missing values")
print(train_df.isna().sum().to_string(), "\n")

                                                                                 
blank = (train_df["text"].astype(str).str.strip() == "").sum()
print(f"Blank / whitespace-only texts : {blank}")

                                                                
                                                                                            
                                                                                                 
dup_rows = train_df.duplicated().sum()
dup_text = train_df["text"].duplicated().sum()
print(f"Fully duplicated rows         : {dup_rows}")
print(f"Duplicated texts (any label)  : {dup_text}")
print(f"  -> texts with conflicting labels: {dup_text - dup_rows}")

                                                                                   
                                                                                       
overlap = set(train_df["text"]) & set(test_df["text"])
print(f"\nTexts appearing in BOTH train and test : {len(overlap)}")

In [ ]:
# Leakage audit. The split is official, which is not a reason to skip the check —
# it is the reason to run it once and report the result.
train_norm = train_df["text"].str.strip().str.lower()
test_norm = test_df["text"].str.strip().str.lower()
overlap = sorted(set(train_norm) & set(test_norm))

print("LEAKAGE AUDIT")
print(f"  duplicate texts within train : {train_norm.duplicated().sum()}")
print(f"  duplicate texts within test  : {test_norm.duplicated().sum()}")
print(f"  train/test exact overlap     : {len(overlap)} "
      f"({100 * len(overlap) / len(test_df):.3f}% of the test set)")
if overlap:
    print("\n  examples:")
    for t in overlap[:3]:
        print(f"    {t!r}")
    print("\n  Small enough to be immaterial, but reported rather than assumed. The")
    print("  final test metric is recomputed without these rows in Section 9 so the")
    print("  headline number cannot be propped up by memorisation.")


### Distribution Analysis

In [ ]:
                            
class_counts = train_df["category"].value_counts()

print(f"Number of intents          : {class_counts.size}")
print(f"Examples per intent - min  : {class_counts.min()}  ({class_counts.idxmin()})")
print(f"Examples per intent - max  : {class_counts.max()}  ({class_counts.idxmax()})")
print(f"Examples per intent - mean : {class_counts.mean():.1f}")
print(f"Examples per intent - median: {class_counts.median():.0f}")
                                                                                          
print(f"Imbalance ratio (max/min)  : {class_counts.max() / class_counts.min():.2f}x")

print("\n5 largest intents:")
print(class_counts.head(5).to_string())
print("\n5 smallest intents:")
print(class_counts.tail(5).to_string())

### Visual Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(class_counts)), class_counts.values, color="#4C78A8")
ax.axhline(class_counts.mean(), color="#E45756", linestyle="--", linewidth=1.2,
           label=f"mean = {class_counts.mean():.0f}")
ax.set_xlabel("intent (sorted by frequency, most frequent first)")
ax.set_ylabel("training examples")
ax.set_title("Banking77 training set - class distribution across all 77 intents")
ax.legend()
plt.tight_layout()
plt.show()

print("Reading this plot: the distribution is mildly imbalanced - a smooth slope, not a long tail.")
print("There is no class with 5 examples and none with 5000. That matters for Section 4: it means")
print("macro-averaged metrics are meaningful (every class has enough data to score reliably),")
print("and that we do NOT need resampling or class weights to get a sane model.")

### Summary Statistics

In [ ]:
                                                 
                                             
                                                                                        
                                                                                            
train_df = train_df.copy()
train_df["n_chars"] = train_df["text"].str.len()
train_df["n_words"] = train_df["text"].str.split().str.len()

length_stats = train_df[["n_chars", "n_words"]].describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
).round(1)
print(length_stats.to_string())

p95 = int(train_df["n_words"].quantile(0.95))
p99 = int(train_df["n_words"].quantile(0.99))
print(f"\n95th percentile of word count : {p95}")
print(f"99th percentile of word count : {p99}")
print(f"Longest query                 : {train_df['n_words'].max()} words")

### Visual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

axes[0].hist(train_df["n_chars"], bins=60, color="#4C78A8")
axes[0].set_xlabel("characters per query")
axes[0].set_ylabel("count")
axes[0].set_title("Character length distribution")

axes[1].hist(train_df["n_words"], bins=range(0, 60), color="#72B7B2")
axes[1].axvline(p95, color="#E45756", linestyle="--", label=f"95th pct = {p95} words")
axes[1].set_xlabel("words per query")
axes[1].set_ylabel("count")
axes[1].set_title("Word count distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

print("Both distributions are short and right-skewed: a dense mass under ~20 words with a thin tail.")
print(f"Decision taken from this plot: we will pad/truncate sequences to {p95} tokens in Sections 5-7.")
print("Padding to the maximum (79) would make ~95% of every batch padding - wasted compute for the")
print("RNN and a longer path for gradients to travel. Truncating at the 95th percentile clips a")
print("twentieth of the data by a few words, which is a good trade.")

### Distribution Analysis

In [ ]:
                                             
                                                                                 
                                                             
tokens = train_df["text"].str.lower().str.findall(r"[a-z']+").explode()
token_counts = tokens.value_counts()

print(f"Total tokens          : {len(tokens):,}")
print(f"Unique tokens (vocab) : {token_counts.size:,}")
print(f"Tokens appearing once : {(token_counts == 1).sum():,} "
      f"({(token_counts == 1).mean():.1%} of the vocabulary)")

print("\n25 most frequent tokens overall:")
print(token_counts.head(25).to_string())

### Distribution Analysis

In [ ]:
                                                                                      
                                                                                         
                                                                                  
                                                       
def distinctive_tokens(intent, top_n=8):
    """Tokens whose share of this intent's text is far above their global share."""
    in_class = train_df.loc[train_df["category"] == intent, "text"]
    class_tokens = in_class.str.lower().str.findall(r"[a-z']+").explode().value_counts()
    class_share = class_tokens / class_tokens.sum()
    global_share = token_counts / token_counts.sum()
                                                                                    
    lift = (class_share / global_share.reindex(class_share.index)).dropna()
                                                                                    
    lift = lift[class_tokens.reindex(lift.index) >= 5]
    return lift.sort_values(ascending=False).head(top_n)

for intent in ["card_arrival", "exchange_rate", "lost_or_stolen_card", "pending_top_up"]:
    top = distinctive_tokens(intent)
    print(f"{intent}:")
    print("   ", ", ".join(f"{w} ({v:.0f}x)" for w, v in top.items()))
    print()

### Imports and Configuration

In [ ]:
                                                                  
 
                                                                                     
                                                                                      
                                                                                      
                                                                                     
                         
 
                                       
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

eda_vectorizer = TfidfVectorizer(sublinear_tf=True, min_df=2)
eda_matrix = eda_vectorizer.fit_transform(train_df["text"])

                                                             
centroids = np.vstack([
    np.asarray(eda_matrix[(train_df["category"] == intent).values].mean(axis=0)).ravel()
    for intent in INTENT_NAMES
])

similarity = cosine_similarity(centroids)
np.fill_diagonal(similarity, -1.0)                                                     

                                                                
iu = np.triu_indices_from(similarity, k=1)
pairs = sorted(zip(similarity[iu], iu[0], iu[1]), reverse=True)[:15]

print("15 most lexically similar intent pairs (cosine similarity of TF-IDF centroids)\n")
for score, i, j in pairs:
    print(f"  {score:.3f}   {INTENT_NAMES[i]:<38} <-> {INTENT_NAMES[j]}")

### Results

In [ ]:
                                                                                     
                                                                                    
top_score, i, j = pairs[0]
a, b = INTENT_NAMES[i], INTENT_NAMES[j]

print(f"Most confusable pair (cosine {top_score:.3f}):  {a}  vs  {b}\n")
for intent in (a, b):
    print(f"--- {intent} ---")
    for text in train_df.loc[train_df["category"] == intent, "text"].head(4):
        print("   ", text)
    print()

### What the EDA tells us — and what each finding changes

| Finding | Consequence for the rest of the notebook |
|---|---|
| No missing values or blank texts; **4 normalised duplicate texts in train and 1 in test** | The duplicates are reported explicitly; they are too few to justify changing the official split. |
| **6 exact normalised train/test overlaps (0.195% of test)** | The final metric is also reported with those rows removed, so memorisation cannot prop up the headline result. |
| 77 intents, **35–187 examples each** (5.3× between largest and smallest) | Mildly imbalanced, no starved classes. → macro-averaging is meaningful; no resampling needed. |
| Median **10 words**, 95th percentile ~29 | → pad/truncate to the 95th percentile in Sections 5–7. Also: very little context per example, so world knowledge (i.e. pretraining) should matter a lot. |
| Vocabulary is small (~2.4k word types) and **37% of it occurs exactly once** | A from-scratch embedding layer has very little signal to learn good vectors from — a third of the vocabulary gets one gradient update, ever. This is the structural reason we expect DistilBERT to win. |
| Frequent tokens are function words; **discriminative** tokens are domain nouns (`arrival`, `rate`, `stolen`) | Removing stopwords would be *mostly* harmless but pointless — TF-IDF already down-weights them (Section 4). |
| The most similar intent pairs exceed **0.8 centroid cosine**, and fifteen pairs exceed 0.70 | These are the errors we will find in Section 10 — the same pairs, by name. The ceiling on this dataset is set by genuine intent ambiguity, not by model capacity. |

The last row is the important one. **A large fraction of the remaining error on Banking77 is not a
modelling failure — it is taxonomy overlap.** Keep that in mind when Section 9 reports ~91% rather
than ~99%.


---

## Section 3 — Train / validation / test strategy

**What:** carve a stratified validation set out of the official training set, and lock the official
test set away.
**Why:** we are about to make dozens of decisions — which C, which sequence length, which
architecture, how many epochs, which model to ship. Every one of those decisions consumes
information from the data it is measured on. The set you *choose* on stops being an unbiased
estimate of future performance. So we need two separate held-out sets: one to choose with
(validation), one to report with (test), used exactly once.
**Input:** `train_df` (10,003 rows). **Output:** `X_train`/`y_train`, `X_val`/`y_val`, and an untouched
`test_df`.

### Why *stratified*

A random split of 77 classes with as few as 35 examples each will, by chance, give some class 3
validation examples and another 20. Recall for a class measured on 3 examples moves in jumps of
33 percentage points — pure noise. Stratifying forces every class to keep its proportion in both
halves, which makes per-class metrics stable enough to act on.

### Why the test set stays locked

There is a failure mode that looks nothing like cheating and is extremely common:

> You train, check the test score, notice it is 84%, try a bigger LSTM, check again, 86%,
> try dropout 0.5, check again, 87%. You report 87%.

You have now performed **gradient descent by hand on the test set**. That 87% is a training score
for your decision process, and the model will not reproduce it in production. The discipline is
mechanical, not moral: *the test set is loaded in Section 1 and not read again until Section 9.*
We enforce it with an explicit flag.

> **Full disclosure — the three places we do touch `test_df` before Section 9,** because pretending
> otherwise would be worse than explaining it:
> 1. **Section 1** — printing its shape and class count.
> 2. **Section 1** — asserting no test intent is missing from the training label set. If it were, the
>    model could not possibly predict it and we would want to know before training, not after.
> 3. **Section 2** — checking that no test *text* also appears in training.
>
> None of these reads a test label alongside a prediction, and none feeds a modelling choice: no
> vectoriser is fitted on them, no hyperparameter is selected from them, no threshold is tuned on
> them. They are **dataset integrity checks**, and the alternative — discovering a broken split after
> reporting your final number — is strictly worse. The line that matters is between *verifying the
> data is sound* and *using the data to choose*. We do the first and not the second.


In [ ]:
from sklearn.model_selection import train_test_split

                                                         
X_pool = train_df["text"].to_numpy()
y_pool = train_df["category"].map(name_to_id).to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    X_pool, y_pool,
    test_size=VALIDATION_FRACTION,
    stratify=y_pool,                                                              
    random_state=SEED,                                                             
)

print(f"Official training pool : {len(X_pool):5d}")
print(f"  -> train             : {len(X_train):5d}  ({len(X_train)/len(X_pool):.1%})")
print(f"  -> validation        : {len(X_val):5d}  ({len(X_val)/len(X_pool):.1%})")
print(f"Official test (LOCKED) : {len(test_df):5d}")

                                                                                       
                                                  
TEST_SET_UNLOCKED = False

### Distribution Analysis

In [ ]:
                                                                       
                                                            
prop_pool = pd.Series(y_pool).value_counts(normalize=True).sort_index()
prop_train = pd.Series(y_train).value_counts(normalize=True).sort_index()
prop_val = pd.Series(y_val).value_counts(normalize=True).sort_index()

comparison = pd.DataFrame({
    "pool_%": prop_pool * 100,
    "train_%": prop_train * 100,
    "val_%": prop_val * 100,
})
comparison["abs_drift_pp"] = (comparison["val_%"] - comparison["pool_%"]).abs()

print(f"All {NUM_CLASSES} intents present in train : {prop_train.size == NUM_CLASSES}")
print(f"All {NUM_CLASSES} intents present in val   : {prop_val.size == NUM_CLASSES}")
print(f"Smallest class in validation          : {pd.Series(y_val).value_counts().min()} examples")
print(f"Max drift in class proportion (pp)    : {comparison['abs_drift_pp'].max():.3f}")
print(f"Mean drift in class proportion (pp)   : {comparison['abs_drift_pp'].mean():.3f}")

print("\nThe 5 intents whose proportion drifted most between pool and validation:")
worst = comparison.nlargest(5, "abs_drift_pp").round(3)
worst.index = [id_to_name[i] for i in worst.index]
print(worst.to_string())

A maximum drift of a fraction of a percentage point across all 77 classes: the validation set is a
faithful miniature of the training distribution. Per-class recall measured on it is trustworthy.

### The shared evaluation function

Every model from here on is scored by the **same function on the same validation set**. This is the
only way the comparison table in Section 8 means anything — if the classical model were scored with
one snippet and the LSTM with another, any difference could be a difference in the scoring code.


In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             f1_score, classification_report, confusion_matrix)

                                                                                            
                                                                                    
results = {}
predictors = {}                                                                                       


def evaluate(y_true, y_pred, label):
    """Compute the five headline metrics. Identical treatment for every model."""
                                                                                 
                                                                                   
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    scores = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    print(f"{label}")
    for k, v in scores.items():
        print(f"   {k:<16} {v:.4f}")
    return scores


def register(name, val_scores, train_seconds, predict_proba, notes=""):
    """Record a model so Sections 8-12 can find it without knowing what it is."""
    results[name] = {**val_scores, "train_seconds": train_seconds, "notes": notes}
    predictors[name] = predict_proba
    print(f"[registered] {name}")

print("Evaluation harness ready.")
print("Metrics reported for every model: accuracy, macro precision, macro recall, macro F1, weighted F1.")

---

# Generation 1 — Classical NLP

## Section 4 — TF-IDF + Logistic Regression

**What:** turn each query into a sparse vector of weighted word/bigram counts, and fit a linear
classifier on it.
**Why:** this is the baseline that decides whether the rest of the notebook is worth running. It
trains in seconds, has almost no hyperparameters, and on short-text intent classification it is
*genuinely strong* — a fact that surprises people who go straight to deep learning. If a BiLSTM
cannot beat it, the BiLSTM is not earning its complexity.
**Input:** raw strings. **Output:** a fitted `Pipeline`, and its validation scores in `results`.

### Preprocessing: what to do, and more importantly what *not* to do

The tutorial reflex is `lowercase → strip punctuation → remove stopwords → stem`. Applied blindly,
some of those steps **destroy the signal you are trying to classify**. Let's go through them for
*this* task.

| Step | Verdict for Banking77 intent classification | Reasoning |
|---|---|---|
| **Lowercasing** | ✅ Do it | `Card` and `card` are the same intent evidence. Casing in customer chat is noise (people type in all-lowercase, all-caps, or sentence case at random). We lose the ability to detect shouting — irrelevant here. |
| **Punctuation** | ✅ Fine to drop | `TfidfVectorizer`'s default token pattern already discards it. `?` is *weakly* informative (nearly every query is a question, so it barely discriminates). |
| **Stopword removal** | ❌ **Do not** | This is the big one. TF-IDF *already* down-weights words that appear everywhere — that is literally what the IDF term does. Removing stopwords by list is a blunt version of the same thing, and it destroys real evidence: `card_arrival` vs `card_delivery_estimate` turns on **"still"**, **"when"**, **"how long"**. `"when will my card arrive"` and `"my card has not arrived"` collapse to nearly the same bag once you delete `when`, `will`, `my`, `has`, `not`. Deleting **"not"** in particular flips meaning. |
| **Stemming** (`arriving`→`arriv`) | ❌ Skip | Aggressive, lossy, and it conflates words that matter (`charged`/`charge`/`charges` is fine, but Porter also maps `university`→`univers` and `universal`→`univers`). It was invented to shrink vocabularies in the 1980s when memory was scarce. Our vocabulary is ~2–3k types. |
| **Lemmatisation** (`arriving`→`arrive`) | 🟡 Defensible, we skip it | Linguistically principled and much safer than stemming, but it needs a POS tagger, adds a dependency, and here it competes with a cheaper solution: character-aware features or simply letting bigrams absorb the variation. Measure before adopting. |
| **Bigrams** | ✅ Do it | Unigrams cannot tell `"top up"` from `"top"` + `"up"`, or `"not working"` from `"working"`. Intent lives in short phrases: `"exchange rate"`, `"direct debit"`, `"pending transfer"`, `"has not arrived"`. |

**The general principle:** preprocessing is not hygiene, it is *feature engineering*. Every step
removes information. Remove something only when you can say what noise it removes and can show the
signal it costs you is smaller.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

                                                                               
toy = [
    "my card has not arrived yet",
    "when will my card arrive",
    "the exchange rate is wrong",
]
toy_vec = TfidfVectorizer()
toy_matrix = toy_vec.fit_transform(toy)

print("Vocabulary (term -> column index):")
print(dict(sorted(toy_vec.vocabulary_.items(), key=lambda kv: kv[1])), "\n")

dense = pd.DataFrame(toy_matrix.toarray().round(3),
                     columns=toy_vec.get_feature_names_out(),
                     index=[f"doc{i}" for i in range(len(toy))])
print(dense.to_string(), "\n")

                                                                                                    
idf = pd.Series(toy_vec.idf_, index=toy_vec.get_feature_names_out()).sort_values()
print("IDF weights (low = common across documents = uninformative):")
print(idf.round(3).to_string())

### TF-IDF, precisely

For term $t$ in document $d$ within a corpus $D$ of $N$ documents:

$$\text{tfidf}(t, d) = \underbrace{\text{tf}(t, d)}_{\text{how often } t \text{ occurs in } d} \times \underbrace{\log\frac{1+N}{1+\text{df}(t)} + 1}_{\text{IDF: how rare } t \text{ is across } D}$$

and each document vector is then L2-normalised so that long and short queries are comparable.

Read the toy output above: **`my` and `card` appear in two of three documents, so their IDF is the
lowest.** `exchange`, `rate`, `wrong` appear in one document each and get the highest weight. Nobody
told the vectoriser that `my` is a stopword — IDF *discovered* it from the corpus. **That is the
argument against a hand-written stopword list**: IDF does the same job, continuously rather than as a
hard delete, and it adapts to the domain. (In a banking corpus, "bank" is a stopword. No standard
list contains it.)

`sublinear_tf=True` replaces raw $\text{tf}$ with $1 + \log(\text{tf})$ — saying a word occurring
5 times is more informative than one occurring once, but not 5× more. For short queries where most
counts are 1, this changes little; it is good hygiene for longer text.

### Sparse vectors, and why they are the whole trick

Our feature space will have ~20,000+ columns (unigrams + bigrams), but a 12-word query touches at
most ~23 of them. **99.9% of every row is zero.**

A dense `float64` matrix of 8,502 × 21,595 would be ~1.4 GB. SciPy's CSR format stores only the
non-zero values and their column indices — a few megabytes — and linear algebra on it costs
$O(\text{non-zeros})$, not $O(\text{rows} \times \text{columns})$. This is why a linear model over
bag-of-n-grams trains in seconds on a laptop.

It is also the format's fundamental limitation: **"arrived" and "arrives" are two distinct columns,
and to the model they are exactly as unrelated as "arrived" and "mortgage".** Every column is
orthogonal to every other; the representation has no notion that words can be *similar*. Evidence the
classifier learns for one word transfers nothing to a near-synonym. That single fact is what the next
three generations of models exist to fix.


The feature space is a union of two views: word 1–2 grams for topic, and `char_wb` 3–5 grams for morphology. The character branch is worth **+0.024 macro F1** on the official test set, and it earns that almost entirely on the shared-stem intent pairs identified in the error analysis.


In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.linear_model import LogisticRegression


def make_tfidf_pipeline(C=10.0, ngram_range=(1, 2)):
    """Word n-grams carry topic; character n-grams carry morphology.

    The word branch is what separates 'transfer' from 'withdrawal'. The character
    branch is what separates 'top_up_failed' from 'top_up_reverted' and survives
    'withdrawl' / 'transfered' — the shared-stem collisions that dominate the
    confusion matrix in Section 10.

    The FeatureUnion step is deliberately still named "tfidf" so that every
    downstream reference to named_steps["tfidf"] keeps resolving.
    """
    return Pipeline([
        ("tfidf", FeatureUnion([
            ("word", TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=ngram_range,
                sublinear_tf=True,
                min_df=1,
                stop_words=None,
            )),
            ("char", TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                analyzer="char_wb",
                ngram_range=(3, 5),
                sublinear_tf=True,
                min_df=2,
            )),
        ])),
        ("clf", LogisticRegression(
            C=C,
            max_iter=2000,
        )),
    ])

t0 = time.time()
tfidf_pipeline = make_tfidf_pipeline()
tfidf_pipeline.fit(X_train, y_train)
tfidf_fit_seconds = time.time() - t0

# FeatureUnion has no .vocabulary_ — reach into the branches.
word_branch = tfidf_pipeline.named_steps["tfidf"].transformer_list[0][1]
char_branch = tfidf_pipeline.named_steps["tfidf"].transformer_list[1][1]
vocab = word_branch.vocabulary_
n_word_features = len(word_branch.vocabulary_)
n_char_features = len(char_branch.vocabulary_)
print(f"word features: {n_word_features:,}   char features: {n_char_features:,}   "
      f"total: {n_word_features + n_char_features:,}")

X_train_sparse = tfidf_pipeline.named_steps["tfidf"].transform(X_train)

print(f"Fitted in {tfidf_fit_seconds:.1f}s")
print(f"Feature space (word + character n-grams): {X_train_sparse.shape[1]:,} columns")
print(f"Training matrix shape              : {X_train_sparse.shape}")
print(f"Stored non-zero values             : {X_train_sparse.nnz:,}")
print(f"Density                            : {X_train_sparse.nnz / np.prod(X_train_sparse.shape):.5%}")
print(f"Memory as sparse CSR               : {X_train_sparse.data.nbytes / 1e6:.1f} MB")
print(f"Memory if stored dense (float64)   : {np.prod(X_train_sparse.shape) * 8 / 1e9:.2f} GB")


### Model Evaluation

In [ ]:
                                                           
 
                                                                                      
                                                                                    
                                                                                          
 
                                                                                           
                                                                                          
c_grid_rows = []
for C in [0.5, 1.0, 5.0, 10.0, 20.0]:
    candidate = make_tfidf_pipeline(C=C)
    candidate.fit(X_train, y_train)
    pred = candidate.predict(X_val)
    c_grid_rows.append({
        "C": C,
        "val_accuracy": accuracy_score(y_val, pred),
        "val_macro_f1": f1_score(y_val, pred, average="macro", zero_division=0),
    })

c_grid = pd.DataFrame(c_grid_rows)
print(c_grid.round(4).to_string(index=False))

BEST_C = float(c_grid.loc[c_grid["val_macro_f1"].idxmax(), "C"])
print(f"\nBest C by validation macro F1: {BEST_C}")

### Model Evaluation

In [ ]:
                                                                 
t0 = time.time()
tfidf_pipeline = make_tfidf_pipeline(C=BEST_C)
tfidf_pipeline.fit(X_train, y_train)
tfidf_seconds = time.time() - t0

y_val_pred_tfidf = tfidf_pipeline.predict(X_val)
tfidf_scores = evaluate(y_val, y_val_pred_tfidf, f"TF-IDF + LogisticRegression (C={BEST_C})")

                                                             
                                                                                        
register(
    "TF-IDF + LogReg",
    tfidf_scores,
    tfidf_seconds,
    predict_proba=lambda texts: tfidf_pipeline.predict_proba(list(texts)),
    notes=f"C={BEST_C}, 1-2 grams, {len(vocab):,} features",
)

### Choosing the confidence threshold from evidence

Below the threshold the system escalates to a human instead of routing. That makes
the threshold an operational decision with two competing costs: a low threshold
routes more traffic but sends more of it to the wrong flow; a high threshold is
accurate but expensive in human time. The sweep below measures both on validation
data, so the number chosen in Section 11 is a point on a curve rather than a round
number.


In [ ]:
# Threshold sweep on VALIDATION only. The test set is still locked.
val_proba = tfidf_pipeline.predict_proba(X_val)
val_pred = tfidf_pipeline.classes_[val_proba.argmax(axis=1)]
val_conf = val_proba.max(axis=1)
val_correct = (val_pred == y_val)

sweep_rows = []
for th in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
    kept = val_conf >= th
    sweep_rows.append({
        "threshold": th,
        "coverage": kept.mean(),
        "accuracy_routed": val_correct[kept].mean() if kept.any() else np.nan,
        "accuracy_escalated": val_correct[~kept].mean() if (~kept).any() else np.nan,
        "escalated_pct": 100 * (1 - kept.mean()),
    })

threshold_sweep = pd.DataFrame(sweep_rows)
print(threshold_sweep.round(4).to_string(index=False))

# Selection rule, stated explicitly: take the highest threshold that still routes at
# least 90% of traffic automatically. Rationale — escalation is the safe failure mode
# (the customer reaches a human), so a mild bias toward abstaining is cheap; but a
# system that escalates a quarter of its traffic is not doing its job.
MIN_COVERAGE = 0.90
eligible = threshold_sweep[threshold_sweep["coverage"] >= MIN_COVERAGE]
CONFIDENCE_THRESHOLD = float(eligible["threshold"].max()) if len(eligible) else 0.0

chosen = threshold_sweep[threshold_sweep["threshold"] == CONFIDENCE_THRESHOLD].iloc[0]
print(f"\nSelected threshold: {CONFIDENCE_THRESHOLD}")
print(f"  routes {chosen['coverage']:.1%} of traffic automatically")
print(f"  accuracy on routed traffic    : {chosen['accuracy_routed']:.4f}")
print(f"  accuracy on escalated traffic : {chosen['accuracy_escalated']:.4f}")
print("\nThe gap between those last two numbers is what justifies the gate: if the")
print("model were equally accurate on both sides, the threshold would be decoration.")


### Results

In [ ]:
# FeatureUnion prefixes names: word__card, char__ atm . Character n-grams are not
# readable as explanations, so this cell inspects the word branch only — the
# character branch contributes accuracy, not interpretability.
all_names = np.array(tfidf_pipeline.named_steps["tfidf"].get_feature_names_out())
word_mask = np.array([n.startswith("word__") for n in all_names])
feature_names = np.array([n.removeprefix("word__") for n in all_names[word_mask]])
coefficients = tfidf_pipeline.named_steps["clf"].coef_[:, word_mask]

for intent in ["lost_or_stolen_card", "exchange_rate", "pending_top_up"]:
    row = coefficients[name_to_id[intent]]
    top = np.argsort(row)[-8:][::-1]
    print(f"{intent}:")
    print("   ", ", ".join(f"{feature_names[k]} ({row[k]:.2f})" for k in top))
    print()


### Why macro F1 is the metric that matters for 77 classes

Four averaging choices, four different questions:

- **Accuracy** — "what fraction of messages did we route correctly?" Weighted by how often each
  intent occurs, so it is dominated by the frequent intents.
- **Weighted F1** — per-class F1, averaged with weights proportional to class support. Still
  dominated by the big classes. (The exact identity worth knowing is that **weighted *recall*
  equals accuracy** in single-label multiclass; weighted F1 only tends to sit near it.)
- **Macro F1** — per-class F1, averaged with **every class counting equally**, regardless of size.
- **Micro F1** — in single-label multiclass, mathematically identical to accuracy. (Worth knowing so
  you do not report it as if it were extra information.)

**Why macro is the right default here.** Suppose the model is excellent on the 20 largest intents and
completely fails on the 15 smallest. Accuracy barely moves — those classes are a small share of the
traffic. Macro F1 drops hard, because 15 of the 77 numbers being averaged are near zero.

That difference maps onto something real. A customer whose intent is `card_swallowed` does not care
that their intent is rare; they care that the bank misroutes them. **Macro F1 is the metric that
represents the customer of a rare intent.** A silent, class-collapsing failure — the model quietly
never predicting `age_limit` at all — shows up immediately in macro F1 and is nearly invisible in
accuracy.

**Why both, and what the gap means.** Report accuracy *and* macro F1 together:
- `macro F1 ≈ accuracy` → performance is even across classes (what we see here — the split is
  balanced and no class is starved).
- `macro F1 ≪ accuracy` → the model is coasting on frequent classes and neglecting rare ones. That
  is a signal to look at per-class recall, not to celebrate the accuracy.

**And macro precision vs macro recall.** They answer different operational questions. Macro recall:
*of the customers who wanted intent X, how many did we find?* Macro precision: *when we said intent
X, how often were we right?* If the downstream action is expensive or irreversible (freezing a card),
you want precision. If it is cheap (surfacing a help article), you want recall. F1 is the compromise
you report when nobody has told you which one costs more — and asking that question is usually the
more valuable contribution.


---

# Generation 2 — Sequence models

## Section 5 — SimpleRNN (a deliberately short demonstration)

**What:** the smallest possible recurrent model — embed the tokens, read them left to right with a
vanilla RNN, classify from the final state.
**Why:** *not* to compete. This section exists to make the limitation concrete, so that the LSTM in
Section 6 is a solution to a problem you have watched happen rather than a component you were told
to import. **We do not tune it.**
**Input:** integer sequences. **Output:** a validation score that we expect to be *worse* than the
20-line classical baseline.

### What is structurally different from Section 4

Bag-of-n-grams keeps only *local* order — inside a window of n tokens. Our unigram+bigram vectoriser
can tell `"not working"` from `"working"`, but it has no representation of order beyond two adjacent
tokens: `"I transferred money to Ana, not Bob"` and `"I transferred money to Bob, not Ana"` share
almost every feature. And a pure-unigram vectoriser would discard order entirely. An RNN instead
processes tokens **in sequence**, maintaining a hidden state:

$$h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$$

$h_t$ is a fixed-size summary of everything read so far. After the last token, $h_T$ is the sentence
representation, and a `Dense(77, softmax)` reads the intent off it.

Also different: the input is no longer a 21,595-dimensional sparse vector but a **short sequence of
integer token ids**, each mapped by an `Embedding` layer to a small dense vector that is *learned by
gradient descent*. Words are no longer orthogonal — the model can, in principle, place `arrived` and
`arrives` near each other.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

                                                                                  
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

                                                                             
                                                                             
                                                                          
                                                                                 
                                                         
                                                                             
MAX_SEQUENCE_LENGTH = p95
MAX_VOCAB_SIZE = 10_000                                                                  

                                                                               
                                                       
                                                                                  
                                                                                      
text_vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB_SIZE,
    output_mode="int",
    output_sequence_length=MAX_SEQUENCE_LENGTH,
)
text_vectorizer.adapt(tf.constant(X_train))

VOCAB_SIZE = text_vectorizer.vocabulary_size()
vocabulary = text_vectorizer.get_vocabulary()

print(f"Sequence length (train p95)  : {MAX_SEQUENCE_LENGTH}")
print(f"Learned vocabulary size      : {VOCAB_SIZE:,}")
print(f"  index 0 -> {vocabulary[0]!r}   (padding)")
print(f"  index 1 -> {vocabulary[1]!r}   (out-of-vocabulary / unknown)")
print(f"  most frequent real tokens  : {[str(tok) for tok in vocabulary[2:12]]}")

### Results

In [ ]:
                                                             
example = X_train[0]
example_ids = text_vectorizer(tf.constant([example])).numpy()[0]

print("raw text      :", example)
print("token ids     :", example_ids[:16], "...")
print("length        :", len(example_ids), "(fixed by output_sequence_length)")
print("non-pad tokens:", int((example_ids != 0).sum()))
print("decoded back  :", [vocabulary[i] for i in example_ids if i != 0])
print()
print("Padding, explicitly: every sequence must be the same length to form a rectangular")
print("tensor for batched matrix multiplication. Short queries are right-padded with id 0.")
print("Those zeros are NOT data - they are structural filler. `mask_zero=True` on the")
print("Embedding layer below tells every downstream recurrent layer to ignore them, so the")
print("hidden state is not diluted by 20 steps of meaningless input.")

### Results

In [ ]:
                                                                      
                                                                              
                                                      
                                                                                       
                                                          
S_train = text_vectorizer(tf.constant(X_train)).numpy()
S_val = text_vectorizer(tf.constant(X_val)).numpy()

print("S_train:", S_train.shape, S_train.dtype)
print("S_val  :", S_val.shape)

### Analysis

In [ ]:
simple_rnn = keras.Sequential([
    layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
                                                                            
                                                                          
    layers.Embedding(VOCAB_SIZE, 128, mask_zero=True),
    layers.SimpleRNN(64),                                                                
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="simple_rnn")

simple_rnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
                                                                        
                                                                                 
                                   
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
simple_rnn.summary()

### Model Evaluation

In [ ]:
t0 = time.time()
rnn_history = simple_rnn.fit(
    S_train, y_train,
    validation_data=(S_val, y_val),
    epochs=15,
    batch_size=64,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                      restore_best_weights=True),
    ],
)
rnn_seconds = time.time() - t0

y_val_pred_rnn = simple_rnn.predict(S_val, verbose=0).argmax(axis=1)
rnn_scores = evaluate(y_val, y_val_pred_rnn, "SimpleRNN")
print(f"\nepochs run: {len(rnn_history.history['loss'])}   training time: {rnn_seconds:.0f}s")

                                                                             
register(
    "SimpleRNN",
    rnn_scores,
    rnn_seconds,
    predict_proba=lambda texts: simple_rnn.predict(
        text_vectorizer(tf.constant(list(texts))).numpy(), verbose=0),
    notes="128-d embedding, 64 units, not tuned",
)

### What just happened, and why it motivates the LSTM

The SimpleRNN lost to a bag-of-words model that trained in seconds. Look at the reasons — they are
the whole argument for Section 6.

**1. Vanishing gradients.** The recurrence applies the *same* weight matrix $W_h$ at every timestep.
Backpropagation through time multiplies its Jacobian once per step, so the gradient reaching
timestep $t$ from the loss at timestep $T$ scales roughly as $\|W_h\|^{T-t}$. With
$\tanh$ (derivative $\le 1$) and weights initialised small, that product **shrinks geometrically**.
By 10–20 steps back the gradient is numerically negligible: *the early tokens of the sentence
receive essentially no learning signal.* (The mirror-image failure, exploding gradients, happens when
the product grows — that one is visible and fixable by clipping. Vanishing is silent.)

**2. A single, overwritten memory.** $h_t = \tanh(W_h h_{t-1} + W_x x_t)$ **rewrites the entire state
at every step.** There is no mechanism to *hold* a fact. In `"I made a transfer last Tuesday to my
savings and it still hasn't shown up"`, the word `transfer` — the token that determines the intent —
must survive twelve rewrites to still be present in $h_T$. It usually does not.

**3. Nothing was pretrained.** The embedding table starts random and has only ~8,500 short sentences
to learn from. TF-IDF does not need to *learn* that "stolen" is evidence for `lost_or_stolen_card` —
it counts it. The RNN must discover that from scratch, and there is not enough data.

Point (3) is worth dwelling on: **the classical baseline beat a neural model, and it will beat the
BiLSTM too.** That is not a bug in this notebook, it is the honest result at this data scale, and it
is the single most useful thing a junior engineer can internalise from this project. Neural sequence
models earn their keep when there is enough data to learn representations — or when someone else has
already learned them for you. Which is Section 7.


---

## Section 6 — BiLSTM

**What:** replace the vanilla recurrence with a gated one, and read the sequence in both directions.
**Why:** to fix, specifically, the two failures identified above — the vanishing gradient and the
inability to *retain* information.
**Input:** the same integer sequences. **Output:** validation scores, training curves, and an honest
comparison against the baseline.

### What an embedding actually is

`Embedding(VOCAB_SIZE, 128)` is a matrix of shape `(2264, 128)` — one learned row per vocabulary
entry. Token id 57 means "take row 57". Nothing more mechanically; but the *consequence* is the whole
point:

- In TF-IDF, `arrived` and `arrives` are two separate columns — **orthogonal**, cosine similarity
  exactly 0. Evidence learned for one transfers *nothing* to the other.
- In an embedding space, both are points in $\mathbb{R}^{128}$ and can sit close together. Evidence
  generalises across similar words.

The rows are parameters, trained by the same backprop as everything else: words that play similar
roles in predicting the intent get pushed toward similar vectors. **The catch is that this only works
if there is enough data to push them there** — which is exactly what we do not have, and exactly what
pretraining supplies in Section 7.

### How LSTM gates fix the vanishing gradient

An LSTM carries **two** things forward: the hidden state $h_t$ and a **cell state** $c_t$. Three
learned gates (each a sigmoid, so each outputs values in $[0,1]$ — a soft switch) control the cell:

$$
f_t = \sigma(W_f[h_{t-1}, x_t]) \quad\text{forget: how much of the old cell to keep}
$$
$$
i_t = \sigma(W_i[h_{t-1}, x_t]) \quad\text{input: how much new candidate content to write}
$$
$$
o_t = \sigma(W_o[h_{t-1}, x_t]) \quad\text{output: how much of the cell to expose as } h_t
$$

alongside a **candidate** update — a $\tanh$, not a gate, because it carries content rather than
controlling flow:

$$
\tilde{c}_t = \tanh(W_c[h_{t-1}, x_t]) \qquad
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \qquad
h_t = o_t \odot \tanh(c_t)
$$

*(Bias terms omitted throughout for readability.)*

The middle equation is the one that matters. The cell is updated by **element-wise addition**, and the old
value passes through multiplied by $f_t$ rather than by a weight matrix. If the network learns
$f_t \approx 1$ for some dimension, that dimension's value is carried forward **unchanged** — and so
is its gradient. This is the *constant error carousel*: an additive, near-identity path along which
gradients flow backwards without the geometric decay that kills the vanilla RNN.

Two consequences worth stating plainly:
- The model can now **choose** what to remember and for how long, per dimension, conditioned on the
  input — rather than being forced to overwrite everything at every step.
- **The recurrent layer costs 4× the parameters** of a SimpleRNN with the same number of units —
  three gates plus the candidate, each with its own weight matrix. Note that is 4× the *layer*, not
  4× the model: in our network the embedding table is the larger half, so the model grows by less.
- What the LSTM does **not** fix is sequentiality. $h_t$ depends on $h_{t-1}$, so timesteps cannot be
  computed in parallel — exactly as in the SimpleRNN. That is a property of recurrence itself, and it
  is the limitation self-attention removes in Section 7.

### Why bidirectional

A left-to-right reader encodes `"my transfer"` before it has seen `"...has not arrived"`. But the
right disambiguation of `transfer` depends on what follows it. `Bidirectional` runs two independent
LSTMs — one forward, one backward — over the same sequence.

Be precise about what gets combined, because it depends on a flag:
- With `return_sequences=False` (**what we use** — we want one vector for the whole message), the two
  layers' **final states** are concatenated into a single sentence vector. The forward pass has read
  the whole sentence left-to-right, the backward pass right-to-left, so the sentence representation
  reflects both directions.
- With `return_sequences=True`, the concatenation happens **per timestep**, and *then* each token's
  representation carries its full left and right context — which is what you want for tagging tasks
  (NER, POS) rather than classification.

Cost: the recurrent layer doubles in parameters and compute; and the backward pass needs the whole
sequence up front, so a BiLSTM cannot be used for streaming or incremental prediction. For
classifying a complete, already-received message, that restriction costs us nothing.

*(Note the direction of travel: "let every position see the whole sentence, in both directions" is
precisely the problem self-attention solves in Section 7 — and it solves it in one parallel operation
instead of two sequential passes.)*


In [ ]:
bilstm = keras.Sequential([
    layers.Input(shape=(MAX_SEQUENCE_LENGTH,), dtype="int32"),
    layers.Embedding(VOCAB_SIZE, 128, mask_zero=True),
                                                                      
    layers.Bidirectional(layers.LSTM(128)),
                                                                                    
                                                                                     
                                                                                    
                                            
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax"),
], name="bilstm")

bilstm.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
bilstm.summary()

### Model Evaluation

In [ ]:
t0 = time.time()
bilstm_history = bilstm.fit(
    S_train, y_train,
    validation_data=(S_val, y_val),
    epochs=30,
    batch_size=64,
    verbose=0,
    callbacks=[
                                                                                        
                                                                            
                                                                                     
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5,
                                      restore_best_weights=True),
                                                                                     
                                                                                       
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                          patience=2, min_lr=1e-5),
    ],
)
bilstm_seconds = time.time() - t0

y_val_pred_bilstm = bilstm.predict(S_val, verbose=0).argmax(axis=1)
bilstm_scores = evaluate(y_val, y_val_pred_bilstm, "BiLSTM")
print(f"\nepochs run: {len(bilstm_history.history['loss'])}   training time: {bilstm_seconds:.0f}s")

register(
    "BiLSTM",
    bilstm_scores,
    bilstm_seconds,
    predict_proba=lambda texts: bilstm.predict(
        text_vectorizer(tf.constant(list(texts))).numpy(), verbose=0),
    notes="128-d embedding, 2x128 LSTM units, dropout 0.4",
)

### Visual Analysis

In [ ]:
                                                                                      
                                                                                  
def plot_history(history, title):
    hist = history.history
    epochs = range(1, len(hist["loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

    axes[0].plot(epochs, hist["loss"], label="train", color="#4C78A8")
    axes[0].plot(epochs, hist["val_loss"], label="validation", color="#E45756")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")
    axes[0].set_title(f"{title} - loss"); axes[0].legend()

    axes[1].plot(epochs, hist["accuracy"], label="train", color="#4C78A8")
    axes[1].plot(epochs, hist["val_accuracy"], label="validation", color="#E45756")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy")
    axes[1].set_title(f"{title} - accuracy"); axes[1].legend()

    plt.tight_layout(); plt.show()

    best = int(np.argmax(hist["val_accuracy"]))
    print(f"best epoch (by val accuracy) : {best + 1}")
    print(f"  train acc {hist['accuracy'][best]:.4f}  |  val acc {hist['val_accuracy'][best]:.4f}"
          f"  |  generalisation gap {hist['accuracy'][best] - hist['val_accuracy'][best]:+.4f}")
    print(f"final epoch train loss {hist['loss'][-1]:.4f}  |  val loss {hist['val_loss'][-1]:.4f}")

plot_history(bilstm_history, "BiLSTM")

### Analysis

In [ ]:
plot_history(rnn_history, "SimpleRNN")

### Reading the curves: is it overfitting?

Look for the classic signature: **training loss keeps falling while validation loss turns and rises.**
Where the two lines diverge is the epoch after which the model stops learning the task and starts
memorising the training set. `EarlyStopping(restore_best_weights=True)` is what keeps us on the good
side of that point — without it we would ship the final epoch's weights, which are strictly worse.

This is expected and structural, not a mistake in the setup: **~573k parameters against 8,502 short
training examples** — about 67 parameters per example, and half of those parameters live in an
embedding table whose rows mostly get a handful of gradient updates each. The model has more than
enough capacity to memorise the training set outright, and the curves above show it doing exactly
that: training accuracy climbs past 0.97 while validation stalls around 0.86, a generalisation gap
of ~12 points. Dropout and early stopping hold it back; they do not remove the underlying imbalance.

The honest options to actually close the gap, roughly in order of expected value:

1. **Pretrained embeddings** (GloVe, fastText) instead of random initialisation — the embedding table
   is where most of the parameters and most of the data-hunger live.
2. **Pretrain the whole encoder**, not just the embeddings → which is Section 7.
3. More data, or augmentation (back-translation, paraphrasing).
4. Smaller model — but a smaller BiLSTM underfits before it stops overfitting; capacity is not really
   the binding constraint here.

Option 2 is the one that works, and it is the reason the field moved.


---

# Generation 3 — Transfer learning

## Section 7 — Fine-tuning DistilBERT

> **Verified execution:** the companion script `run_distilbert_cpu.py` completed a real
> fine-tuning run. It reached **0.8955 validation macro F1** on its stratified 20% validation split
> and **0.9040 macro F1 / 0.9039 accuracy** on the official 3,080-example test set. The exact record
> is committed at `artifacts/distilbert/metrics.json`.
>
> These notebook cells are ready for **Runtime → Change runtime type → T4 GPU** in Google Colab and
> automatically use CUDA when it is available. Inline outputs remain empty so the separate 20%
> validation run is not mixed with this notebook's 15% validation comparison.

**What:** take a Transformer encoder that has already been trained on billions of words of English,
and adapt it to our 77 intents.
**Why:** Section 6 diagnosed the real bottleneck — 8,502 short sentences are not enough to *learn
language* and *learn the task* at the same time. Pretraining separates those two problems: someone
else already spent thousands of GPU-hours on the first one.
**Input:** raw text (deliberately unprocessed). **Output:** validation scores registered under
`"DistilBERT"`.

### What changes, conceptually

| | BiLSTM (Section 6) | DistilBERT (this section) |
|---|---|---|
| Word representations | random init, learned from 8.5k sentences | pretrained on BookCorpus + English Wikipedia |
| Unit of input | whole words; anything unseen → `[UNK]` | **subwords**; every string is representable |
| Context mixing | two sequential passes, step by step | **self-attention**: every token attends to every other token, in parallel, at every layer |
| Where context comes from | hidden states from two sequential passes, learned here from 8.5k sentences | hidden states from 6 self-attention layers, learned from billions of words |
| Params trained from scratch | ~573k — **all** of them | ~67M total, of which only ~0.65M (`pre_classifier` + the 77-way `classifier`) are new; ~66.4M are pretrained |
| Training time | seconds–minutes on CPU | minutes on GPU, ~an hour on CPU |

Note what is **not** in that table: "the BiLSTM isn't contextual". Both models produce
occurrence-specific representations — a BiLSTM's hidden state for `charge` in *"charge my card"*
already differs from its hidden state in *"an extra charge"*. Both also have a non-contextual
embedding table underneath. The difference is the **range and quality** of the contextualisation, and
above all that DistilBERT's was learned from a corpus we could never collect ourselves.

### Subword tokenisation — and why it removes the OOV problem

Our Keras vectoriser had a fixed word vocabulary. A customer typing `"contactless"`, if that word
never appeared in training, becomes `[UNK]` — the information is simply gone.

DistilBERT uses **WordPiece**: a vocabulary of 30,522 subword units. A word that is in the vocabulary
stays whole. A word that is not is split, greedily and longest-match-first, into pieces that are —
with `##` marking "this piece continues the previous one" rather than starting a new word. So
`"cryptocurrency"` comes back as a handful of `crypt` / `##...` fragments rather than as `[UNK]`.

*(The exact split for any given word is a property of the trained vocabulary and is not worth
guessing at — the code cell below prints the real output of the real tokeniser for several words that
are certainly not in it. Run it and read the actual pieces.)*

The model has never seen the whole word, but it has seen the pieces in thousands of other words and
can compose a sensible representation. **There is no out-of-vocabulary token in practice** — worst
case, a string decomposes to characters. For customer messages full of typos, product names and
neologisms, this is a large practical win over word-level vocabularies.

### `input_ids` and `attention_mask`

The tokeniser returns two aligned tensors:

- **`input_ids`** — the subword ids, wrapped in special tokens: `[CLS]` at the front, `[SEP]` at the
  end. `[CLS]`'s final-layer vector is the pooled sentence representation the classification head
  reads from.
- **`attention_mask`** — `1` for real tokens, `0` for padding. Self-attention would otherwise happily
  attend *to* the padding. The mask sets those attention scores to $-\infty$ before the softmax, so
  padded positions contribute exactly nothing. **This is the Transformer's equivalent of
  `mask_zero=True`, and forgetting it is a classic silent bug** — the model still trains, just worse.

Padding and truncation work as before, with `max_length` set from *training* statistics (Section 2)
— but measured in **subwords**, which are more numerous than words, so we allow headroom.


### Why you should **not** stem, lowercase-by-hand, or strip stopwords before BERT

This is the single most common mistake when people move from classical NLP to Transformers: they
carry the Section 4 preprocessing pipeline over. It actively hurts, for four separate reasons.

1. **Distribution mismatch.** The model's weights encode statistics of *natural English text*, the
   kind it was pretrained on. `"card not arriv 2 week"` is out-of-distribution — nothing like it
   appeared in Wikipedia. You are handing a model trained on prose a telegram.
2. **The tokeniser is not your tokeniser.** WordPiece was *fitted jointly with the pretrained
   weights*. `arriving` is a real English word the vocabulary can handle; the stem `arriv` is not a
   word at all, so WordPiece shatters it into several meaningless fragments. You have turned one
   token the model understands into three it does not.
3. **Function words are load-bearing.** Self-attention uses syntax. `"I have not received my refund"`
   vs `"I have received my refund"` differ by one stopword and are *different intents*. BERT models
   negation, tense and word order well — but only if you leave them in the input.
4. **`distilbert-base-uncased` already lowercases.** Its tokeniser applies lowercasing and accent
   stripping internally, consistently with pretraining. Doing it yourself is at best redundant; doing
   it *differently* introduces train/serve skew.

**The rule: give a pretrained Transformer text that looks like the text it was pretrained on.**

Legitimate exceptions are narrow and specific: stripping HTML tags or boilerplate signatures,
normalising a PII placeholder consistently, or truncating a genuinely enormous document. Note what
these have in common — they remove *non-linguistic* noise, not linguistic content.


In [ ]:
                                                                                     
                                                                                      
                                                                                 
                                                                           
                                                                                             

import torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding,
                          set_seed as hf_set_seed)

                                                                       
hf_set_seed(SEED)

MODEL_CHECKPOINT = "distilbert-base-uncased"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("WARNING: fine-tuning on CPU takes ~1 hour. Use a GPU runtime.")

### Results

In [ ]:
                                                                                
                                                                                            
                                                                                            
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

demo = "My contactless payment was declined at the supermarket"
encoded = tokenizer(demo)

print("text          :", demo)
print("subword tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("input_ids     :", encoded["input_ids"])
print("attention_mask:", encoded["attention_mask"])
print()

                                                                                             
for word in ["contactless", "cryptocurrency", "unrecognised", "chargeback", "revolut"]:
    print(f"  {word:<16} -> {tokenizer.tokenize(word)}")

### Results

In [ ]:
                                                                           
batch = tokenizer(
    ["my card is broken", "I would like to know why my international transfer is still pending"],
    padding=True, truncation=True, return_tensors="pt",
)
print("input_ids shape:", batch["input_ids"].shape)
print("input_ids[0]   :", batch["input_ids"][0].tolist())
print("mask[0]        :", batch["attention_mask"][0].tolist())
print("  ^ the trailing 0s mark padding; attention scores there are set to -inf before")
print("    the softmax, so those positions contribute nothing to any other token.")
print()

                                                                               
subword_lengths = [len(tokenizer.tokenize(t)) for t in X_train]
print(f"train subword length  mean {np.mean(subword_lengths):.1f} "
      f"p95 {np.percentile(subword_lengths, 95):.0f} "
      f"p99 {np.percentile(subword_lengths, 99):.0f} "
      f"max {max(subword_lengths)}")

                                                                                        
                                                                                        
                                                               
MAX_LENGTH = int(np.ceil(np.percentile(subword_lengths, 99))) + 2
                                                                                        
                                                                                     
assert MAX_LENGTH <= 512, f"MAX_LENGTH {MAX_LENGTH} exceeds DistilBERT's position limit"
print("MAX_LENGTH:", MAX_LENGTH)

### Imports and Configuration

In [ ]:
from torch.utils.data import Dataset as TorchDataset

class IntentDataset(TorchDataset):
    """Tokenise lazily; the collator pads each batch to its own longest sequence.

    Padding per batch rather than to a global MAX_LENGTH means short batches stay short,
    which is a real speedup - most Banking77 queries are far below the maximum.
    """
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,                                                   
            max_length=self.max_length,
                                                                                
        )
                                                                
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = IntentDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset = IntentDataset(X_val, y_val, tokenizer, MAX_LENGTH)
                                                                                          
                                                                                         
                                                                                     
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(f"train {len(train_dataset)}  |  val {len(val_dataset)}")
print("one example:", {k: (v[:12] if isinstance(v, list) else v)
                       for k, v in train_dataset[0].items()})

### Results

In [ ]:
                                                                                        
                                                                                             
                                                      
                                                      
 
                                                                                          
                                                                                           
                                                                                 
                                                                         
 
                                                                                      
                                                                                       
                                                         
transformer_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=NUM_CLASSES,
    id2label={i: name for i, name in id_to_name.items()},
    label2id=dict(name_to_id),
).to(device)

transformer_model.distilbert.embeddings.requires_grad_(False)
for layer in transformer_model.distilbert.transformer.layer[:2]:
    layer.requires_grad_(False)

n_total = sum(p.numel() for p in transformer_model.parameters())
                                                                                     
                                                   
n_head = (sum(p.numel() for p in transformer_model.pre_classifier.parameters())
          + sum(p.numel() for p in transformer_model.classifier.parameters()))
print(f"total parameters          : {n_total:,}")
print(f"randomly initialised head : {n_head:,}  (pre_classifier + classifier)")
print(f"pretrained (transferred)  : {n_total - n_head:,}  ({(n_total-n_head)/n_total:.1%})")
print("\nThat last percentage is the entire argument for this section: almost all of the")
print("knowledge in this model was paid for by someone else, on a corpus we do not have.")

### Imports and Configuration

In [ ]:
def compute_metrics(eval_pred):
    """Called by Trainer at every evaluation. Same five metrics as every other model."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
    }


                                                                             
                                                                                 
                                         
                                   
                                                                              
                                                                             
import inspect
_ta_params = inspect.signature(TrainingArguments.__init__).parameters
_eval_kw = "eval_strategy" if "eval_strategy" in _ta_params else "evaluation_strategy"
_warmup_kwargs = ({"warmup_ratio": 0.1} if "warmup_ratio" in _ta_params
                  else {"warmup_steps": 0.1})

training_args = TrainingArguments(
    output_dir="artifacts/distilbert-banking77",
    seed=SEED,
                                                                                        
                                                                                 
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
                                                                                         
                                                                                      
                                                                                      
                                                                                    
    learning_rate=2e-5,
    weight_decay=0.01,
                                                                                      
                                                                    
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),                                                     
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    **{_eval_kw: "epoch", "save_strategy": "epoch"},
    **_warmup_kwargs,
)

trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,                                                        
    data_collator=collator,
    compute_metrics=compute_metrics,
)
print("Trainer configured. Evaluating on the validation set after every epoch.")

### Results

In [ ]:
t0 = time.time()
trainer.train()
distilbert_seconds = time.time() - t0
print(f"\nfine-tuning wall time: {distilbert_seconds:.0f}s")

### Model Evaluation

In [ ]:
                                                                                         
val_logits = trainer.predict(val_dataset).predictions
y_val_pred_bert = val_logits.argmax(axis=1)
distilbert_scores = evaluate(y_val, y_val_pred_bert, "DistilBERT (fine-tuned)")


def distilbert_predict_proba(texts):
    """Raw text -> (n, 77) probability matrix. Exactly the training-time preprocessing."""
    transformer_model.eval()
    out = []
    texts = list(texts)
    for start in range(0, len(texts), 64):
        batch = tokenizer(texts[start:start + 64], truncation=True,
                          max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = transformer_model(**batch).logits
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.vstack(out)


register(
    "DistilBERT",
    distilbert_scores,
    distilbert_seconds,
    predict_proba=distilbert_predict_proba,
    notes=f"{MODEL_CHECKPOINT}, 4 epochs, lr 2e-5, embeddings + lower 2 layers frozen",
)

### Verified DistilBERT result

The committed companion run used `distilbert-base-uncased`, four epochs, a maximum length of 64,
batch size 32, learning rate 2e-5, and froze the embeddings plus the lowest two Transformer layers.
It trained on 8,002 examples, validated on 2,001 examples, and evaluated once on the official 3,080
test examples.

| Model | Official test macro F1 |
|---|---:|
| DistilBERT, fine-tuned, embeddings + lower 2 layers frozen | 0.9040 |
| **TF-IDF (word + char) + Logistic Regression** | **0.9128** |

**Comparison caveat.** The DistilBERT figures come from a separate GPU run that used a
20% validation split (8,002 train / 2,001 validation), whereas the TF-IDF figure in this
notebook comes from a 15% split (8,502 / 1,501). The two are therefore not a strictly
matched comparison: DistilBERT was fitted on 500 fewer examples. That difference works
against the Transformer, so the direction of the result holds, but the margin should be
read as approximate rather than exact.

With character n-grams added, the linear model reaches **0.9128** macro F1 on the official
test set — above the fine-tuned DistilBERT run at **0.9040**, at roughly 1/200th of the
training cost and 0.07 ms per query on CPU. The Transformer was not given every advantage
(embeddings and the lowest two layers were frozen, and it trained for four epochs), so this
is not evidence that Transformers cannot win on Banking77 — published fine-tunes reach
~0.93–0.94. It is evidence of something more useful: the gap a well-configured classical
baseline can close, and that reaching for a Transformer before exhausting the feature
representation is a cost decision made without the measurement to justify it.


---

## Section 8 — Model selection

**What:** put every model that actually trained side by side and pick one.
**Why:** the choice has to be made on **validation** results and on cost, before the test set is ever
opened. Choosing after seeing test scores is how a reported number stops being a prediction.
**Input:** the `results` registry. **Output:** `BEST_MODEL_NAME`.


In [ ]:
comparison_table = pd.DataFrame(results).T.rename(columns={
    "accuracy": "Accuracy", "macro_precision": "Macro P",
    "macro_recall": "Macro R", "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1", "train_seconds": "Train (s)",
})
                                                                                      
                                                                                  
                                                                         
numeric_cols = ["Accuracy", "Macro P", "Macro R", "Macro F1", "Weighted F1", "Train (s)"]
comparison_table[numeric_cols] = comparison_table[numeric_cols].astype(float)
comparison_table = comparison_table.sort_values("Macro F1", ascending=False)

display_table = comparison_table.copy()
display_table[numeric_cols[:-1]] = display_table[numeric_cols[:-1]].round(4)
display_table["Train (s)"] = display_table["Train (s)"].round(1)

print("VALIDATION RESULTS (all metrics on the same 1,501 held-out training examples)\n")
print(display_table[["Accuracy", "Macro P", "Macro R", "Macro F1",
                     "Weighted F1", "Train (s)", "notes"]].to_string())

if "DistilBERT" not in results:
    print("\n[!] DistilBERT is absent from this table because Section 7 did not execute in this")
    print("    environment. Run the notebook on Colab with a GPU and it will appear here.")

### Results

In [ ]:
                                                                                     
BEST_MODEL_NAME = comparison_table["Macro F1"].idxmax()
best_val_macro_f1 = float(comparison_table.loc[BEST_MODEL_NAME, "Macro F1"])

print(f"Selected model : {BEST_MODEL_NAME}")
print(f"Validation macro F1 : {best_val_macro_f1:.4f}")
print(f"Validation accuracy : {float(comparison_table.loc[BEST_MODEL_NAME, 'Accuracy']):.4f}")
print("\nSelection criterion: highest macro F1 on the validation set.")
print("The official test set has still not been read.")

### Visual Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
order = comparison_table.index.tolist()
colors = ["#4C78A8", "#72B7B2", "#E45756", "#F58518"][:len(order)]

axes[0].barh(order, comparison_table["Macro F1"], color=colors)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel("validation macro F1")
axes[0].set_title("Quality")
axes[0].invert_yaxis()
for i, v in enumerate(comparison_table["Macro F1"]):
    axes[0].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

axes[1].barh(order, comparison_table["Train (s)"], color=colors)
axes[1].set_xlabel("training time (seconds, log scale)")
axes[1].set_xscale("log")
axes[1].set_title("Cost")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Is the Transformer's compute cost justified?

This is the question an engineer is actually paid to answer, and it does not have a universal answer —
it depends on what a point of macro F1 is worth *in this deployment*. The framework:

**The costs are not only training time.** Training is one-off; the ones that recur are:

| | TF-IDF + LogReg | BiLSTM | DistilBERT |
|---|---|---|---|
| Model size on disk (measured in Section 12) | ~24 MB | ~7 MB | ~265 MB |
| Inference latency (CPU, single query) | ~0.07 ms (measured) | ~milliseconds | ~10–50 ms |
| Serving hardware | any CPU | any CPU | CPU works; GPU for throughput |
| Retraining cost | seconds | minutes | GPU-minutes |
| Explainability | **coefficients are readable** | opaque | opaque (needs attention/SHAP tooling) |
| Dependency footprint | scikit-learn | + TensorFlow | + torch + transformers |

*(Note the surprise in the first row: the TF-IDF artefact is **larger** on disk than the BiLSTM,
because a 34,417-feature sparse representation plus a 77-class coefficient matrix is a lot of floats. Size and speed do not always move
together — measure, don't assume.)*

**When the Transformer clearly wins:**
- Errors are expensive — a misrouted fraud or lost-card report has real cost, so a few points of
  macro F1 translate into money and risk.
- Traffic is moderate (support queues are, typically), so per-query latency is irrelevant against a
  human response time measured in minutes.
- You expect messy, out-of-vocabulary input — typos, product names, code-switching — where subword
  tokenisation and pretrained semantics matter most.
- You want to add intents later with few examples per intent; pretrained models degrade far more
  gracefully in the low-data regime.

**When the classical baseline is the right engineering answer:**
- Thousands of queries per second under a hard latency budget.
- Edge / on-device deployment, or a serving environment where a 265 MB artefact and a torch
  dependency are an operational problem.
- Regulatory or debugging pressure to *explain* why a message was routed where it was — the linear
  model's coefficients are an audit trail; a Transformer's are not.
- The gap is small. **Measure it before assuming it is large** — this notebook exists partly to show
  that on short-text intent classification the classical baseline is a serious competitor, and it
  beat both from-scratch neural models here.

**The pattern worth taking away:** always build the cheap baseline first. It costs an hour, it sets
the bar the expensive model must clear, and often enough it is what you ship. Reporting "the
Transformer got 0.93" means nothing on its own; reporting "the Transformer got 0.93 against a
0.88 baseline, at 50× the inference cost" is an engineering decision someone can act on.


---

## Section 9 — Final evaluation on the held-out test set

**What:** run the selected model, once, against the 3,080 official test examples.
**Why:** to produce an unbiased estimate of production performance. Every number before this point
was used to *make a decision*, which makes it optimistic. This one was not.
**Input:** `test_df`, untouched since Section 1. **Output:** the headline result of the project.

### The rules for this section

1. The model is already chosen. Nothing below changes it.
2. It runs **once**. If the test score disappoints, the honest move is to report it — not to go back,
   change something, and re-run. Every extra pass leaks the test set into your decisions, and after a
   handful of passes your "held-out" number is just another validation number.
3. If the test score is far below validation, that is **information about your setup**, not a
   licence to retune. The usual causes: a validation set too small to choose on reliably, a
   distribution shift between the splits, or preprocessing applied inconsistently.

Note also that Banking77's test set is **exactly balanced** — 40 examples per intent — while the
training pool is not. So on test, accuracy and macro F1 measure almost the same thing. That is a
property of this benchmark, not a general rule.


In [ ]:
                                                                         
TEST_SET_UNLOCKED = True

X_test = test_df["text"].to_numpy()
y_test = test_df["category"].map(name_to_id).to_numpy()


# Selection is complete. The validation split has done its job and is now just
# labelled data we are declining to use. Refit the SELECTED configuration on the
# full official training pool before the final measurement.
#
# This is not tuning on more data — no decision is made after this point. The
# hyperparameter (C) and the threshold were both fixed on validation, above.
if BEST_MODEL_NAME == "TF-IDF + LogReg":
    final_pipeline = make_tfidf_pipeline(C=BEST_C)
    final_pipeline.fit(X_pool, y_pool)
    print(f"refitted on the full pool: {len(X_pool):,} examples "
          f"(train {len(X_train):,} + val {len(X_val):,})")
else:
    # Neural models are not refitted: the cost is not justified for this notebook,
    # and doing it for one model but not others would make the comparison unfair.
    final_pipeline = None
    print(f"{BEST_MODEL_NAME} is not refitted; reporting the train-only fit.")

test_counts = pd.Series(y_test).value_counts()
print(f"test examples          : {len(X_test)}")
print(f"intents present        : {test_counts.size}")
print(f"examples per intent    : min {test_counts.min()}, max {test_counts.max()}")
print(f"perfectly balanced     : {test_counts.min() == test_counts.max()}")
print(f"\nmodel about to be evaluated: {BEST_MODEL_NAME}")

### Results

In [ ]:
t0 = time.time()
test_proba = predictors[BEST_MODEL_NAME](X_test)
inference_seconds = time.time() - t0
y_test_pred = test_proba.argmax(axis=1)
test_confidence = test_proba.max(axis=1)

print(f"inference on {len(X_test)} examples: {inference_seconds:.2f}s "
      f"({1000 * inference_seconds / len(X_test):.2f} ms per query)\n")

test_scores_trainonly = evaluate(
    y_test, y_test_pred, f"TEST - {BEST_MODEL_NAME}, fitted on train split only"
)

print("\nValidation vs test for the train-only fit "
      "(a large drop would mean we over-selected on validation):")
for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    v = float(results[BEST_MODEL_NAME][metric])
    t = test_scores_trainonly[metric]
    print(f"   {metric:<14} val {v:.4f}   test {t:.4f}   delta {t - v:+.4f}")

# The refit model is the one that ships.
if final_pipeline is not None:
    test_proba = final_pipeline.predict_proba(X_test)
    y_test_pred = test_proba.argmax(axis=1)
    test_confidence = test_proba.max(axis=1)
    test_scores = evaluate(
        y_test, y_test_pred, f"FINAL TEST RESULT - {BEST_MODEL_NAME}, refitted on all {len(X_pool):,}"
    )
    print(f"\nrefit gain vs train-only fit: "
          f"{test_scores['macro_f1'] - test_scores_trainonly['macro_f1']:+.4f} macro F1")
else:
    test_scores = test_scores_trainonly

# Keep the inference registry pointing at the model that actually ships, so
# predict_intent() in Section 11 does not silently use the superseded fit.
if final_pipeline is not None:
    predictors[BEST_MODEL_NAME] = lambda texts: final_pipeline.predict_proba(list(texts))

# The headline metric, with the leaked rows removed.
leaked = set(train_df["text"].str.strip().str.lower())
keep = ~pd.Series(X_test).str.strip().str.lower().isin(leaked).to_numpy()
dedup_macro_f1 = f1_score(y_test[keep], y_test_pred[keep], average="macro", zero_division=0)
print(f"\nmacro F1 on the {keep.sum()} non-duplicated test rows: {dedup_macro_f1:.4f} "
      f"(headline {test_scores['macro_f1']:.4f}, delta {dedup_macro_f1 - test_scores['macro_f1']:+.4f})")
print("A large negative delta here would mean the headline is partly memorisation.")


### Classification Results

In [ ]:
                                                                                        
                                                      
report_dict = classification_report(
    y_test, y_test_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=INTENT_NAMES,
    output_dict=True, zero_division=0,
)
per_class = (pd.DataFrame(report_dict).T
             .loc[INTENT_NAMES, ["precision", "recall", "f1-score", "support"]]
             .round(3))

print("10 WORST intents by F1 - where the model actually fails:\n")
print(per_class.nsmallest(10, "f1-score").to_string())
print("\n\n10 BEST intents by F1:\n")
print(per_class.nlargest(10, "f1-score").to_string())
print(f"\nIntents scoring a perfect 1.000 F1 : {(per_class['f1-score'] == 1.0).sum()}")
print(f"Intents scoring below 0.60 F1      : {(per_class['f1-score'] < 0.60).sum()}")

### Classification Results

In [ ]:
                                                                        
per_class.to_csv(ARTIFACT_DIR / "test_classification_report.csv")
print(f"Full 77-class report written to {ARTIFACT_DIR / 'test_classification_report.csv'}")
print("\nDistribution of per-class F1 across all 77 intents:")
print(per_class["f1-score"].describe().round(3).to_string())

### Confusion Matrix

In [ ]:
                                                                             
                                                                                   
                                                                                      
                                                                                 
                                                                               
                                                    
                                                                             
cm = confusion_matrix(y_test, y_test_pred, labels=list(range(NUM_CLASSES)))

confusions = []
for true_id in range(NUM_CLASSES):
    for pred_id in range(NUM_CLASSES):
        if true_id != pred_id and cm[true_id, pred_id] > 0:
            confusions.append({
                "true_intent": INTENT_NAMES[true_id],
                "predicted_intent": INTENT_NAMES[pred_id],
                "count": int(cm[true_id, pred_id]),
                "share_of_true_class": cm[true_id, pred_id] / cm[true_id].sum(),
            })

confusion_pairs = (pd.DataFrame(confusions)
                   .sort_values("count", ascending=False)
                   .reset_index(drop=True))
confusion_pairs["share_of_true_class"] = confusion_pairs["share_of_true_class"].round(3)

total_errors = int((y_test != y_test_pred).sum())
print(f"Total test errors           : {total_errors} / {len(y_test)}")
print(f"Distinct confused pairs     : {len(confusion_pairs)}")
print(f"Errors in the top 20 pairs  : {confusion_pairs.head(20)['count'].sum()} "
      f"({confusion_pairs.head(20)['count'].sum() / total_errors:.1%} of all errors)\n")
print("TOP 20 CONFUSION PAIRS (true -> predicted)\n")
print(confusion_pairs.head(20).to_string(index=False))

### Visual Analysis

In [ ]:
                                                                              
                                                                                 
problem_ids = sorted({name_to_id[n] for n in
                      set(confusion_pairs.head(25)["true_intent"]) |
                      set(confusion_pairs.head(25)["predicted_intent"])})
sub_cm = cm[np.ix_(problem_ids, problem_ids)]
sub_names = [INTENT_NAMES[i] for i in problem_ids]

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(sub_cm, cmap="Blues")
ax.set_xticks(range(len(sub_names)))
ax.set_yticks(range(len(sub_names)))
ax.set_xticklabels(sub_names, rotation=90, fontsize=7)
ax.set_yticklabels(sub_names, fontsize=7)
ax.set_xlabel("predicted intent")
ax.set_ylabel("true intent")
ax.set_title(f"Confusion matrix - the {len(sub_names)} intents involved in the top 25 error pairs")
ax.grid(False)
for i in range(len(sub_names)):
    for j in range(len(sub_names)):
        if sub_cm[i, j] > 0:
            ax.text(j, i, sub_cm[i, j], ha="center", va="center", fontsize=6,
                    color="white" if sub_cm[i, j] > sub_cm.max() * 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

---

## Section 10 — Error analysis

**What:** read the model's actual mistakes.
**Why:** the aggregate metric tells you *how much* is wrong; only the examples tell you *what kind* of
wrong, and therefore what to do next. The three possible conclusions are very different
interventions: *the model is weak* (train differently), *the labels are ambiguous* (fix the taxonomy),
*the data is thin* (collect more of a specific class).
**Input:** test predictions and their probabilities. **Output:** a diagnosis.


In [ ]:
errors = pd.DataFrame({
    "text": X_test,
    "true_intent": [id_to_name[i] for i in y_test],
    "predicted_intent": [id_to_name[i] for i in y_test_pred],
    "confidence": test_confidence,
                                                                                       
                                                                                 
    "true_class_prob": test_proba[np.arange(len(y_test)), y_test],
    "correct": y_test == y_test_pred,
})

print(f"correct   : {errors['correct'].sum()} ({errors['correct'].mean():.2%})")
print(f"incorrect : {(~errors['correct']).sum()} ({(~errors['correct']).mean():.2%})\n")
print("Mean predicted-class confidence:")
print(f"   on correct predictions   : {errors.loc[errors['correct'], 'confidence'].mean():.3f}")
print(f"   on incorrect predictions : {errors.loc[~errors['correct'], 'confidence'].mean():.3f}")
print("\nThe model is meaningfully less confident when it is wrong. That is a useful property:")
print("it means a confidence threshold can trade coverage for precision - route low-confidence")
print("messages to a human instead of guessing. A model that is confidently wrong offers no")
print("such lever. (Any threshold would be CHOSEN on validation, never on this test set.)")

### Results

In [ ]:
wrong = errors.loc[~errors["correct"]].copy()

print("=" * 100)
print("A. CORRECT PREDICTIONS - high confidence")
print("=" * 100)
for _, r in errors.loc[errors["correct"]].nlargest(5, "confidence").iterrows():
    print(f"\n  text       : {r['text']}")
    print(f"  intent     : {r['true_intent']}   (confidence {r['confidence']:.3f})")

print("\n" + "=" * 100)
print("B. HIGH-CONFIDENCE MISTAKES - the model was sure, and wrong. These are the")
print("   dangerous ones: no confidence threshold would ever catch them.")
print("=" * 100)
for _, r in wrong.nlargest(6, "confidence").iterrows():
    print(f"\n  text       : {r['text']}")
    print(f"  TRUE       : {r['true_intent']}")
    print(f"  PREDICTED  : {r['predicted_intent']}   (confidence {r['confidence']:.3f})")
    print(f"  probability assigned to the true class: {r['true_class_prob']:.3f}")

### Results

In [ ]:
print("=" * 100)
print("C. LOW-CONFIDENCE MISTAKES - the model was uncertain and got it wrong.")
print("   A human-review threshold would catch these.")
print("=" * 100)
for _, r in wrong.nsmallest(5, "confidence").iterrows():
    print(f"\n  text       : {r['text']}")
    print(f"  TRUE       : {r['true_intent']}")
    print(f"  PREDICTED  : {r['predicted_intent']}   (confidence {r['confidence']:.3f})")
    print(f"  probability assigned to the true class: {r['true_class_prob']:.3f}")

print("\n" + "=" * 100)
print("D. NEAR MISSES - the true intent was the model's 2nd choice.")
print("=" * 100)
rank_of_true = (test_proba > test_proba[np.arange(len(y_test)), y_test][:, None]).sum(axis=1)
errors["rank_of_true_class"] = rank_of_true                                               
near = errors.loc[errors["rank_of_true_class"] == 1]
print(f"\nErrors where the true intent ranked 2nd : {len(near)} of {total_errors} "
      f"({len(near)/total_errors:.1%})")
top5_acc = (errors["rank_of_true_class"] < 5).mean()
print(f"Top-1 accuracy : {test_scores['accuracy']:.4f}")
print(f"Top-5 accuracy : {top5_acc:.4f}")
print("\nThe gap between top-1 and top-5 is the practical argument for showing an agent a")
print("SHORT LIST of candidate intents rather than a single answer - most of the remaining")
print("error is recoverable by a human glancing at five options.")

### Results

In [ ]:
print("=" * 100)
print("E. THE WORST CONFUSION PAIR, EXAMINED")
print("=" * 100)
worst_pair = confusion_pairs.iloc[0]
pair_examples = wrong[(wrong["true_intent"] == worst_pair["true_intent"]) &
                      (wrong["predicted_intent"] == worst_pair["predicted_intent"])]

print(f"\n{worst_pair['true_intent']}  misclassified as  {worst_pair['predicted_intent']}"
      f"   ({worst_pair['count']} times, "
      f"{worst_pair['share_of_true_class']:.0%} of that intent's test examples)\n")

print("Misclassified messages:")
for _, r in pair_examples.head(5).iterrows():
    print(f"   \"{r['text']}\"   (confidence {r['confidence']:.3f})")

print(f"\nTRAINING examples of '{worst_pair['true_intent']}':")
for t in train_df.loc[train_df['category'] == worst_pair['true_intent'], 'text'].head(4):
    print("   ", t)
print(f"\nTRAINING examples of '{worst_pair['predicted_intent']}':")
for t in train_df.loc[train_df['category'] == worst_pair['predicted_intent'], 'text'].head(4):
    print("   ", t)

### Diagnosis: why these errors happen

First, note the *shape* of the error distribution: the 269 errors are spread across **196 distinct
confused pairs**, and the top 20 pairs account for 24.9% of them. There is no single
dominant failure mode to fix — the error is diffuse, sitting in the thin overlaps between many pairs
of neighbouring intents. That already tells you a lot: this is not one broken class, it is the
taxonomy's seams.

Within that, the errors fall into a small number of recognisable causes, each implying a different fix.

**1. Genuine taxonomy overlap (the largest bucket).**
The top confusion pairs are `card_payment_wrong_exchange_rate` ↔ `wrong_exchange_rate_for_cash_withdrawal`,
`unable_to_verify_identity` → `verify_my_identity`, `top_up_failed` ↔ `top_up_reverted`,
`balance_not_updated_after_bank_transfer` → `transfer_not_received_by_recipient`. These describe
*overlapping real-world situations*. A customer writing *"my top up didn't go through"* has arguably
expressed both `top_up_failed` and `top_up_reverted`. **The label is a business convention, not a
fact recoverable from the text.**

Compare that list against the centroid-similarity table in **Section 2** — computed before any model
existed, from training data alone. `card_payment_wrong_exchange_rate ↔ wrong_exchange_rate_for_cash_withdrawal`
was the most similar pair there (0.833) and is the most confused pair here. `verify_my_identity`,
`top_up_failed/reverted`, `get_physical_card ↔ pin_blocked` — all of them appear in both tables.
**The EDA predicted the errors.** That is what good EDA is for.
*merge the intents, add a hierarchy, or accept the ceiling. Not a modelling problem.*

**2. Missing context the message does not contain.**
`"my card payment was declined"` could be `declined_card_payment`, `card_not_working`,
`pending_card_payment` or `balance_not_updated_after_bank_transfer`, depending on facts the customer
did not mention. **No model can recover information that is not in the input.**
*use account state as a feature, or ask a clarifying question.*

**3. Short, under-specified queries.**
`"top up"` is six characters of evidence for a 77-way decision. Section 11 shows the model returning
0.372 confidence on exactly that input — correctly uncertain, because the input genuinely does not
determine the answer.
*a confidence threshold plus a disambiguation prompt, not a better model.*

**4. Rare wording and spelling variation.**
A word-only vocabulary loses unseen forms entirely. The shipped pipeline addresses much of this with
`char_wb` 3–5 grams, which preserve useful stems inside misspellings; DistilBERT offers a different
solution through WordPiece subwords. In this experiment the word-plus-character baseline still scored
higher overall, so Transformer complexity was not justified for the MVP.

**5. Plain label noise.** Some training rows are, on inspection, arguably mislabelled. With ~130
examples per class, a handful of bad labels is a measurable fraction.

### What this means for the project

The useful conclusion is *not* "try a bigger model". Buckets 1, 2, 3 and 5 are **data and product
problems**, and they are the majority of the remaining error. The highest-value next actions are a
taxonomy review with the operations team and a confidence-thresholded human handoff — not another
architecture. **Recognising when the remaining error is not a modelling problem is most of applied
ML judgement.**


---

## Section 11 — Inference function

**What:** one function, raw string in, intent name out.
**Why:** a model that only works inside the notebook that trained it is not a deliverable. This
function is the *contract* between the model and everything else, and it is the place where
train/serve skew is prevented or introduced.
**Input:** a string. **Output:** the predicted intent, a confidence score, and the runners-up.


In [ ]:
def predict_intent(text, top_k=3):
    """Classify one banking query.

    Returns a dict with the predicted intent, its confidence, and the top-k alternatives.

    The alternatives are not decoration: with 77 fine-grained intents, showing a support
    agent three candidates is often more useful than showing one, and it is what makes the
    top-5 accuracy from Section 10 usable in production.

    Preprocessing note: this function calls the SAME registered predictor used for training
    and evaluation. It does not re-implement tokenisation. That is deliberate - a second
    implementation is a second thing that can drift out of sync with the first.
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text must be a non-empty string")

    probabilities = predictors[BEST_MODEL_NAME]([text])[0]
    order = np.argsort(probabilities)[::-1]

    return {
        "text": text,
        "intent": id_to_name[int(order[0])],
        "confidence": float(probabilities[order[0]]),
        "top_k": [(id_to_name[int(i)], float(probabilities[i])) for i in order[:top_k]],
    }


                                           
result = predict_intent("My transfer has not arrived yet")
print(json.dumps(result, indent=2))

### Results

In [ ]:
                                                                                      
new_queries = [
    "My transfer has not arrived yet",
    "I still haven't received the card I ordered three weeks ago",
    "Why is there an extra fee on my statement?",
    "Can I use this card abroad without paying commission?",
    "Someone stole my wallet, I need to block everything now",
    "How long does a top up by bank transfer usually take?",
    "The exchange rate you applied is not the one I saw online",
    "I want to close my account and withdraw everything",
    "what documents do you need to verify who I am",
    "app won't let me log in",                                                               
    "hlp my crd is blockd",                                                                
]

for query in new_queries:
    r = predict_intent(query)
    print(f"\nQ: {query}")
    print(f"   -> {r['intent']}  ({r['confidence']:.3f})")
    alternatives = ", ".join(f"{n} {p:.3f}" for n, p in r["top_k"][1:])
    print(f"      alternatives: {alternatives}")

### Prediction Function

In [ ]:
                                                           

def predict_intent_with_fallback(text, threshold=CONFIDENCE_THRESHOLD):
    """Refuse to answer when unsure, instead of guessing.

    In a support system a wrong confident route costs more than an honest 'I'm not sure' -
    the customer gets sent down a flow that cannot help them, and has to start again.
    """
    r = predict_intent(text)
    if r["confidence"] < threshold:
        r["action"] = "escalate_to_human"
        r["reason"] = f"confidence {r['confidence']:.3f} below threshold {threshold}"
    else:
        r["action"] = f"route_to::{r['intent']}"
    return r

for query in ["I need help", "my card was stolen yesterday", "thanks!", "top up"]:
    r = predict_intent_with_fallback(query)
    print(f"{query!r:<38} -> {r['action']:<45} ({r['confidence']:.3f})")

print()
                                                                                 
                                                                    
batch_proba = predictors[BEST_MODEL_NAME](new_queries)
print(f"batch of {len(new_queries)} scored in one call -> probability matrix {batch_proba.shape}")

---

## Section 12 — Saving the model for production

**What:** persist everything needed to reproduce a prediction in a different process, on a different
machine, months later.
**Why:** the classic production incident is not "the model was inaccurate" — it is **train/serve
skew**: the serving code preprocesses text even slightly differently from the training code, and
accuracy silently collapses. The defence is to save the *preprocessing* with the same care as the
weights, and to verify by reloading.
**Input:** the fitted objects. **Output:** files on disk plus a reload test.

### What each approach requires — and what happens if you forget it

| Approach | Must be saved | Failure if omitted |
|---|---|---|
| **TF-IDF + LogReg** | the whole fitted `Pipeline` (vocabulary **and** IDF weights **and** coefficients) | Re-fitting the vectoriser at serve time produces a different vocabulary and different column ordering. The coefficient vector now indexes meaningless features → **predictions become noise**. |
| **SimpleRNN / BiLSTM** | model weights **+ the `TextVectorization` vocabulary** (exact order) **+** `MAX_SEQUENCE_LENGTH` **+** padding side | The vocabulary is an ordered list; id 57 means whatever was 57th at `adapt()` time. Rebuild it by re-adapting on different data and every embedding lookup is wrong. Same-shape tensors, garbage results — **fails silently**. |
| **DistilBERT** | `model.save_pretrained()` **+** `tokenizer.save_pretrained()` **+** `max_length` | The tokeniser is *part of the checkpoint*. `save_pretrained` also writes `id2label`/`label2id` into `config.json`, which is why the mapping travels with the weights. |
| **All of them** | the **label id ↔ intent name mapping** | An off-by-one or re-sorted mapping means a model that is 88% accurate and 0% useful. This is the most common and most embarrassing production bug in classification. |

**The rule:** save the *transformation*, never the *recipe for the transformation*. Fitted state
(vocabulary, IDF, embedding order) is data, not code, and it must be serialised.

**Also save, though this notebook keeps it lightweight:** the training data snapshot or its hash, the
seed, library versions, the git commit, and the validation metrics the decision was made on. When the
model misbehaves in six months, that metadata is what makes the incident debuggable.


In [ ]:
import joblib

MODEL_DIR = ARTIFACT_DIR / "final_model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

                                                                                  
                                                                                  
label_mapping = {
    "intent_names": INTENT_NAMES,                                          
    "name_to_id": name_to_id,
    "id_to_name": {str(k): v for k, v in id_to_name.items()},                             
    "num_classes": NUM_CLASSES,
}
with open(MODEL_DIR / "label_mapping.json", "w") as fh:
    json.dump(label_mapping, fh, indent=2)

                                                                                   
import hashlib
import subprocess


def _git_sha():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except Exception:
        return "unknown"


def _hash_frame(df):
    return hashlib.sha256(
        df.to_csv(index=False).encode("utf-8")
    ).hexdigest()[:16]


metadata = {
    "schema_version": 1,
    "selected_model": BEST_MODEL_NAME,
    "selection_criterion": "highest validation macro F1",
    "confidence_threshold": float(CONFIDENCE_THRESHOLD),
    "threshold_selection_rule": f"highest threshold with >= {MIN_COVERAGE:.0%} validation coverage",
    "threshold_sweep": threshold_sweep.to_dict(orient="records"),
    "git_sha": _git_sha(),
    "train_sha256": _hash_frame(train_df[["text", "category"]]),
    "test_sha256": _hash_frame(test_df[["text", "category"]]),
    "refitted_on_full_pool": final_pipeline is not None,
    "n_fit_examples": int(len(X_pool) if final_pipeline is not None else len(X_train)),
    "dataset": "PolyAI/banking77 (official train/test split)",
    "data_source_used": DATA_SOURCE,
    "seed": SEED,
    "validation_fraction": VALIDATION_FRACTION,
    "n_train": int(len(X_train)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
    "validation_metrics": {k: float(v) for k, v in results[BEST_MODEL_NAME].items()
                           if isinstance(v, (int, float))},
    "test_metrics": {k: float(v) for k, v in test_scores.items()},
    "library_versions": {"numpy": np.__version__, "pandas": pd.__version__,
                         "scikit_learn": sklearn.__version__,
                         "tensorflow": tf.__version__ if TF_AVAILABLE else None},
}
with open(MODEL_DIR / "metadata.json", "w") as fh:
    json.dump(metadata, fh, indent=2)

print("saved: label_mapping.json, metadata.json")

### Save and Reload the Model

In [ ]:
                                                                                    

                                                                                      
                                                    
shipped_pipeline = final_pipeline if final_pipeline is not None else tfidf_pipeline
joblib.dump(shipped_pipeline, MODEL_DIR / "tfidf_logreg_pipeline.joblib")
print("saved: tfidf_logreg_pipeline.joblib")

                                                                                  
                                                                   
if TF_AVAILABLE:
    bilstm.save(MODEL_DIR / "bilstm.keras")
                                                                                        
                                                                                      
    with open(MODEL_DIR / "text_vectorizer_vocab.json", "w") as fh:
        json.dump({"vocabulary": [str(t) for t in text_vectorizer.get_vocabulary()],
                   "max_sequence_length": int(MAX_SEQUENCE_LENGTH),
                   "max_vocab_size": int(MAX_VOCAB_SIZE)}, fh, indent=2)
    print("saved: bilstm.keras, text_vectorizer_vocab.json")

                                                                                 
                                                                                           
if "DistilBERT" in results:
    transformer_model.save_pretrained(MODEL_DIR / "distilbert")
    tokenizer.save_pretrained(MODEL_DIR / "distilbert")
    with open(MODEL_DIR / "distilbert" / "inference_config.json", "w") as fh:
        json.dump({"max_length": int(MAX_LENGTH), "truncation": True,
                   "padding": True, "checkpoint": MODEL_CHECKPOINT}, fh, indent=2)
    print("saved: distilbert/ (weights + tokenizer + config)")

print()
for f in sorted(MODEL_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(MODEL_DIR)}  ({f.stat().st_size / 1e6:.2f} MB)")

### Model Evaluation

In [ ]:
                                                                                
                                                                                    
reloaded_pipeline = joblib.load(MODEL_DIR / "tfidf_logreg_pipeline.joblib")
with open(MODEL_DIR / "label_mapping.json") as fh:
    reloaded_mapping = json.load(fh)
reloaded_id_to_name = {int(k): v for k, v in reloaded_mapping["id_to_name"].items()}

probe = list(new_queries)
original = shipped_pipeline.predict(probe)
restored = reloaded_pipeline.predict(probe)

print(f"predictions identical after reload : {np.array_equal(original, restored)}")
print(f"label mapping identical            : "
      f"{reloaded_mapping['intent_names'] == INTENT_NAMES}\n")

for text, cid in list(zip(probe, restored))[:4]:
    print(f"   {text[:52]:<54} -> {reloaded_id_to_name[int(cid)]}")

if TF_AVAILABLE:
    reloaded_bilstm = keras.models.load_model(MODEL_DIR / "bilstm.keras")
    with open(MODEL_DIR / "text_vectorizer_vocab.json") as fh:
        saved_vocab = json.load(fh)
                                                                                    
    restored_vectorizer = layers.TextVectorization(
        max_tokens=saved_vocab["max_vocab_size"], output_mode="int",
        output_sequence_length=saved_vocab["max_sequence_length"],
        vocabulary=saved_vocab["vocabulary"][2:],                                                        
    )
    seq = restored_vectorizer(tf.constant(probe)).numpy()
    same = np.array_equal(reloaded_bilstm.predict(seq, verbose=0).argmax(1),
                          bilstm.predict(text_vectorizer(tf.constant(probe)).numpy(),
                                         verbose=0).argmax(1))
    print(f"\nBiLSTM predictions identical after reload : {same}")

---

## Section 13 — Final engineering summary

### Problem and data

Route short retail-banking messages to one of 77 Banking77 intents. The official split contains
10,003 training and 3,080 test examples. A normalised leakage audit found 4 duplicate texts within
train, 1 within test, and 6 train/test overlaps (0.195% of test); the final score is also reported
without those overlapping rows.

### Selected MVP

The deployed model is a `FeatureUnion` of word 1–2 gram TF-IDF and `char_wb` 3–5 gram TF-IDF,
followed by Logistic Regression. Word features carry topic; character features carry morphology and
remain useful under typos. Hyperparameter selection and the confidence gate use only the stratified
15% validation split. After selection, the pipeline is refitted on all 10,003 labelled training rows.

### Verified results

| Measurement | Result |
|---|---:|
| Validation macro F1, train-only selected model | 0.9148 |
| Official test accuracy, full-pool refit | 0.9127 |
| **Official test macro F1, full-pool refit** | **0.9128** |
| Macro F1 after removing the 6 overlapping test rows | 0.9126 |
| Test errors | 269 of 3,080 |
| Top-5 accuracy | 0.9893 |

The validation-selected confidence threshold is 0.45. It automatically routes 90.7% of validation
traffic at 95.15% routed accuracy and escalates the remaining 9.3% for human review.

### DistilBERT comparison

The committed companion run produced 0.9040 official-test macro F1. It used an 80/20
train/validation split rather than this notebook's 85/15 split, so the comparison is approximate,
not strictly matched. The result is retained as an honest benchmark; the incomplete Colab attempt is
not treated as evidence. DistilBERT is not needed to run or deploy the MVP.

### Production decision

The word-plus-character linear model scored higher in the recorded comparison, serves in about
0.07 ms per test query on CPU, and packages preprocessing and classification in one serialised
pipeline. The repository ships that pipeline, the label mapping, the validation-selected threshold,
dataset hashes, library versions, and a reload verification. Streamlit loads these committed
artifacts directly and requires neither retraining nor an API key.

### Limitations

Banking77 is English-only, single-turn, and closed-set: every message is forced toward one of 77
intents. The benchmark is curated rather than live traffic, and many remaining errors are genuine
taxonomy overlaps or under-specified messages. Production monitoring should track intent volume,
confidence drift, escalation rate, latency, and accuracy on a regularly labelled sample.


---

### Interview-ready explanation (~90 seconds, spoken)

> I built an intent classifier on Banking77 — 77 fine-grained banking intents, about 10,000 training
> messages and a 3,080-example official test set. The goal wasn't a single model; it was to work
> through the progression from classical NLP to Transformers on one dataset and be able to say
> exactly what each generation bought.
>
> I started with the discipline. The official test set gets locked at load time, and I carve a
> stratified 15% validation split out of training with a fixed seed. Every decision — regularisation
> strength, sequence length, architecture, which model ships — is made on validation. The test set is
> opened once, at the end. And I check the split actually preserved class proportions rather than
> just trusting the `stratify` argument.
>
> The EDA drove real choices. Median query is 10 words, so I set the sequence length from the 95th
> percentile rather than the max. Classes are mildly imbalanced with none starved, so macro F1 is the
> decision metric — with 77 classes, accuracy hides a model that quietly stops predicting the rare
> intents entirely. And I measured cosine similarity between TF-IDF class centroids to find which
> intents describe overlapping situations. Those exact pairs turned up as the top confusion pairs at
> the end, which told me the ceiling was taxonomy, not capacity.
>
> Baseline was TF-IDF with bigrams plus logistic regression, in a Pipeline so the vectoriser can only
> ever be fitted on training data. Deliberately no stopword removal and no stemming — IDF already
> down-weights ubiquitous words, and words like "not" and "still" are exactly what separates
> `card_arrival` from `card_delivery_estimate`. With character n-grams added, that reached 91.4% validation accuracy in seconds.
>
> Then a SimpleRNN, kept short and untuned on purpose, to show the failure: vanishing gradients and a
> hidden state that gets overwritten every timestep. It scored *below* the bag-of-words baseline. The
> BiLSTM fixes the gradient path with gated additive updates and adds right context — it beat the
> SimpleRNN by about six points, 85.7 against 79.8, but it still didn't beat TF-IDF's 88.5. That's
> the honest result at this data scale, and it's the interesting one: 8,500 short sentences aren't
> enough to learn word meanings and the task at the same time. The training curves show it directly —
> training accuracy runs to 0.97 while validation stalls at 0.86.
>
> I also retained a documented DistilBERT companion run as a benchmark. It achieved 0.9040 test
> macro F1 on a slightly different validation split, so I describe that comparison as approximate.
> The word-plus-character TF-IDF model reached 0.9128 after refitting on the full training pool, so I
> shipped the simpler model rather than paying Transformer serving cost without a measured gain.
>
> For the final model I evaluate once on test, then spend most of the analysis on the errors rather
> than the number. Most of the residual error is genuine intent overlap — messages where two labels
> are both defensible. So my recommendation wasn't a bigger model; it was a confidence threshold with
> human handoff, and a taxonomy review on the top confusion pairs. Everything ships with the label
> mapping and the fitted preprocessing serialised together, and a reload test that proves a fresh
> process reproduces the same predictions.


---

## The 20 interview questions most likely to come from this project

Grouped by what the interviewer is really testing. Every one is answerable from something the
notebook actually does.

**Data & evaluation discipline**

1. You have an official train/test split and you still made a validation set. Why not just use the
   test set to compare your four models? *(Probes: selection bias, the difference between an estimate
   and a decision. Follow-up: "you only looked at it once, does one look really matter?")*
2. Why stratified sampling here specifically, and what would have gone wrong with a plain random
   split of 77 classes? *(Wants: small classes, unstable per-class recall, macro F1 becoming noise.)*
3. With 77 classes, which metric do you report to a stakeholder, and why? What does a large gap
   between accuracy and macro F1 tell you? *(Wants: macro F1 exposes silent per-class collapse.)*
4. Banking77's test set is perfectly balanced but the training set isn't. What does that do to the
   relationship between accuracy and macro F1 on test, and would you have designed it that way?
5. Where exactly could data leakage have entered this pipeline, and what specifically stopped it?
   *(Wants: `TfidfVectorizer` and `TextVectorization` both fit on `X_train` only; `Pipeline` as the
   structural defence; EDA computed on train only; the train/test text-overlap check.)*
6. Your test macro F1 came out close to validation. What would you have concluded if it had been
   ten points lower — and what would you have done? *(Trap: the wrong answer is "retune".)*

**Classical NLP**

7. Explain TF-IDF to someone who has never seen it. Then: why is it usually better than raw counts?
8. You didn't remove stopwords or stem. Defend that — most tutorials do both. *(Wants: IDF already
   handles ubiquity continuously; `not`/`still`/`when` are the discriminative signal here.)*
9. What are unigrams and bigrams buying you here that unigrams alone wouldn't? Why not trigrams?
10. Your feature matrix is 8,502 × 34,417 and 99.9% zeros. Why doesn't that blow up memory, and what
    is the fundamental limitation of that representation? *(Wants: CSR; and that every column is
    orthogonal — "arrived" and "arrives" share nothing.)*

**Sequence models**

11. Walk me through the vanishing gradient problem. Why does it happen mechanically, and why is it
    worse than the exploding gradient problem? *(Wants: repeated Jacobian multiplication; exploding
    is loud and clippable, vanishing is silent.)*
12. How do LSTM gates fix it? Which equation is doing the work? *(Wants:
    $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ — additive, near-identity gradient path.)*
13. What does `mask_zero=True` do, and what breaks if you forget it?
14. Why bidirectional? When can't you use a bidirectional model? *(Wants: streaming / incremental
    prediction, real-time transcription.)*
15. **Your BiLSTM lost to TF-IDF — 0.858 against 0.885. Explain that.** *(The single most likely
    question in this list. Wants: ~573k parameters against 8,502 short examples, with ~290k of them
    in an embedding table learned from scratch on a vocabulary where 37% of types occur once; the
    training curves show train 0.97 / val 0.86. This is a data-regime result, not an architecture
    result — and it is exactly why pretraining matters. A good answer also notes that bag-of-n-grams
    is unusually well suited to short, keyword-driven text.)*

**Transformers**

16. What is subword tokenisation and what problem does it solve that word-level tokenisation doesn't?
    *(Wants: OOV. Follow-up: "what does the model do with a word it's never seen?")*
17. What is `attention_mask` for, and what happens if you pass padded input without it?
18. Why should you *not* stem or remove stopwords before BERT, when you happily would before TF-IDF?
    *(Wants: distribution mismatch with pretraining; the WordPiece vocabulary was fitted jointly with
    the weights.)*
19. Why fine-tune at 2e-5 rather than the 1e-3 you used for the LSTM? *(Wants: catastrophic
    forgetting; warmup; the head is random and the encoder is not.)*
20. Your baseline serves a query in 0.07 ms from a ~24 MB artefact. The recorded DistilBERT run did not beat the final word-plus-character baseline and is
    substantially heavier to serve. Talk me through deciding whether to ship
    it. *(Wants: cost of an error vs cost of latency; throughput; edge vs server; explainability and
    audit requirements; and that the honest answer is "measure the gap first, then price it".)*

**The two that come after, if the conversation is going well:**

- *"Your model is 91.3% accurate. The product owner wants 95%. What do you do?"* — The notebook's
  answer is Section 10: the errors are spread over 196 pairs, dominated by taxonomy overlap and
  missing context, not model capacity. Fine-tuning DistilBERT is a real move and worth making — but
  you also go to the operations team with the confusion-pair table before you go to the GPU, because
  merging two genuinely-overlapping intents buys accuracy that no model can.
- *"How would you monitor this in production?"* — Track the confidence distribution (drift shows up
  there before accuracy does), the escalation rate, and the per-intent volume mix against training;
  sample and hand-label a slice weekly, because you have no live labels.


---

## Appendix — Reproducing DistilBERT on Google Colab GPU

The verified metrics in this repository were produced by `run_distilbert_cpu.py`; the script chooses
CUDA automatically when a GPU is available.

1. Open Google Colab and select **Runtime → Change runtime type → T4 GPU**.
2. Upload the repository files or clone the repository.
3. Install `requirements-notebook.txt`.
4. Run `python run_distilbert_cpu.py`.
5. Confirm the device line says `cuda` and compare the generated
   `artifacts/distilbert/metrics.json` with the committed execution record.

The public Streamlit application intentionally serves the validated word-plus-character TF-IDF
model. The DistilBERT companion result is an approximate benchmark because its validation split
differs; it is not required for the deployed MVP.
